In [2]:
import pandas as pd
import numpy as np

# -------------------------
# 1. LOAD RAW MONEYPUCK DATA
# -------------------------

all_teams_path = "all_teams.csv"   # adjust if needed
mp_raw = pd.read_csv(all_teams_path)

print("Raw shape:", mp_raw.shape)
mp_raw.head()


Raw shape: (221100, 111)


,team,season,name,gameId,playerTeam,opposingTeam,home_or_away,gameDate,position,situation,...,unblockedShotAttemptsAgainst,scoreAdjustedUnblockedShotAttemptsAgainst,dZoneGiveawaysAgainst,xGoalsFromxReboundsOfShotsAgainst,xGoalsFromActualReboundsOfShotsAgainst,reboundxGoalsAgainst,totalShotCreditAgainst,scoreAdjustedTotalShotCreditAgainst,scoreFlurryAdjustedTotalShotCreditAgainst,playoffGame
0,NYR,2008,NYR,2008020001,NYR,T.B,AWAY,20081004,Team Level,other,...,1.0,1.000,0.0,0.017,0.000,0.000,0.037,0.037,0.037,0
1,NYR,2008,NYR,2008020001,NYR,T.B,AWAY,20081004,Team Level,all,...,31.0,30.369,5.0,0.396,0.168,0.168,2.917,2.833,2.714,0
2,NYR,2008,NYR,2008020001,NYR,T.B,AWAY,20081004,Team Level,5on5,...,20.0,19.369,3.0,0.237,0.168,0.168,1.862,1.777,1.665,0
3,NYR,2008,NYR,2008020001,NYR,T.B,AWAY,20081004,Team Level,4on5,...,9.0,9.000,1.0,0.124,0.000,0.000,0.795,0.795,0.789,0
4,NYR,2008,NYR,2008020001,NYR,T.B,AWAY,20081004,Team Level,5on4,...,1.0,1.000,1.0,0.019,0.000,0.000,0.224,0.224,0.224,0


In [4]:
# -------------------------
# 2. FILTER TO SITUATION = "all"
# -------------------------

mp = mp_raw.copy()
mp = mp[mp["situation"].str.lower() == "all"].reset_index(drop=True)

print("After filtering to situation='all':", mp.shape)


After filtering to situation='all': (44220, 111)


In [6]:
# -------------------------
# 3. BASIC CLEANING
# -------------------------

# Standardize column names
mp.columns = [c.strip().replace(" ", "_").replace("%", "Pct") for c in mp.columns]

# Clean team codes (just ensure uppercase)
mp["team"] = mp["team"].str.upper()
mp["opposingTeam"] = mp["opposingTeam"].str.upper()

# Convert date
mp["gameDate"] = pd.to_datetime(mp["gameDate"], errors="coerce")

# Derive a season_start_year if needed
# MoneyPuck already includes 'season' like 2023 for 2023-24
mp["season"] = mp["season"].astype(int)

# Create a unique team-game key
mp["team_game_id"] = (
    mp["gameDate"].dt.strftime("%Y%m%d") + "_" +
    mp["team"] + "_" +
    mp["gameId"].astype(str)
)

mp.head()


,team,season,name,gameId,playerTeam,opposingTeam,home_or_away,gameDate,position,situation,...,scoreAdjustedUnblockedShotAttemptsAgainst,dZoneGiveawaysAgainst,xGoalsFromxReboundsOfShotsAgainst,xGoalsFromActualReboundsOfShotsAgainst,reboundxGoalsAgainst,totalShotCreditAgainst,scoreAdjustedTotalShotCreditAgainst,scoreFlurryAdjustedTotalShotCreditAgainst,playoffGame,team_game_id
0,NYR,2008,NYR,2008020001,NYR,T.B,AWAY,1970-01-01 00:00:00.020081004,Team Level,all,...,30.369,5.0,0.396,0.168,0.168,2.917,2.833,2.714,0,19700101_NYR_2008020001
1,NYR,2008,NYR,2008020003,NYR,T.B,HOME,1970-01-01 00:00:00.020081005,Team Level,all,...,31.984,5.0,0.241,0.000,0.000,1.091,1.117,1.091,0,19700101_NYR_2008020003
2,NYR,2008,NYR,2008020010,NYR,CHI,HOME,1970-01-01 00:00:00.020081010,Team Level,all,...,43.911,3.0,0.448,0.407,0.407,2.738,2.751,2.730,0,19700101_NYR_2008020010
3,NYR,2008,NYR,2008020019,NYR,PHI,AWAY,1970-01-01 00:00:00.020081011,Team Level,all,...,38.025,4.0,0.504,0.401,0.401,3.123,2.958,2.907,0,19700101_NYR_2008020019
4,NYR,2008,NYR,2008020034,NYR,N.J,HOME,1970-01-01 00:00:00.020081013,Team Level,all,...,38.019,3.0,0.383,1.139,1.139,2.698,2.691,2.242,0,19700101_NYR_2008020034


In [8]:
# -------------------------
# 4. CONFIRM PER-TEAM-PER-GAME SHAPE
# -------------------------

print("Unique games:", mp["gameId"].nunique())
print("Unique teams:", mp["team"].nunique())

# Should be ~2 rows per NHL game (one row for each team)
games_per_team = (
    mp.groupby("gameId")["team"]
    .nunique()
)

print("Average teams per gameId:", games_per_team.mean())


Unique games: 22110
Unique teams: 38
Average teams per gameId: 2.0


In [10]:
# -------------------------
# 5. SORT PROPERLY FOR ROLLING WINDOWS LATER
# -------------------------

mp = mp.sort_values(["team", "gameDate"]).reset_index(drop=True)

print("Final prepared shape:", mp.shape)
mp.head()


Final prepared shape: (44220, 112)


,team,season,name,gameId,playerTeam,opposingTeam,home_or_away,gameDate,position,situation,...,scoreAdjustedUnblockedShotAttemptsAgainst,dZoneGiveawaysAgainst,xGoalsFromxReboundsOfShotsAgainst,xGoalsFromActualReboundsOfShotsAgainst,reboundxGoalsAgainst,totalShotCreditAgainst,scoreAdjustedTotalShotCreditAgainst,scoreFlurryAdjustedTotalShotCreditAgainst,playoffGame,team_game_id
0,ANA,2008,ANA,2008020008,ANA,S.J,AWAY,1970-01-01 00:00:00.020081009,Team Level,all,...,48.390,4.0,0.658,0.666,0.666,3.620,3.634,3.428,0,19700101_ANA_2008020008
1,ANA,2008,ANA,2008020030,ANA,ARI,HOME,1970-01-01 00:00:00.020081012,Team Level,all,...,33.943,1.0,0.359,0.899,0.899,1.355,1.437,1.432,0,19700101_ANA_2008020030
2,ANA,2008,ANA,2008020042,ANA,L.A,AWAY,1970-01-01 00:00:00.020081014,Team Level,all,...,35.869,11.0,0.523,0.180,0.180,4.052,3.971,3.711,0,19700101_ANA_2008020042
3,ANA,2008,ANA,2008020048,ANA,EDM,HOME,1970-01-01 00:00:00.020081015,Team Level,all,...,42.560,1.0,0.439,0.161,0.161,2.454,2.571,2.532,0,19700101_ANA_2008020048
4,ANA,2008,ANA,2008020061,ANA,S.J,HOME,1970-01-01 00:00:00.020081017,Team Level,all,...,51.243,4.0,0.684,0.485,0.542,2.748,2.748,2.716,0,19700101_ANA_2008020061


Rolling Averages

In [13]:
import numpy as np
import pandas as pd

def add_rolling_features_moneypuck(
    mp: pd.DataFrame,
    windows=(1, 3, 5, 10),
    min_games: int = 1,
) -> pd.DataFrame:
    """
    Add rolling features for ALL numeric per-team, per-game stats in the
    MoneyPuck all_teams dataset (filtered to situation='all').

    - Computes rolling means and sums for each numeric column.
    - Grouped by team, ordered by gameDate (and gameId as tie-breaker).
    - Uses shift(1) before rolling to avoid data leakage.

    Parameters
    ----------
    mp : pd.DataFrame
        MoneyPuck team-game data (one row per team per game, situation='all').
    windows : tuple
        Rolling window sizes in games, e.g. (1, 3, 5, 10).
    min_games : int
        Minimum number of prior games required to compute a rolling value.
        If 1, then second game gets a one-game history; first game stays NaN.

    Returns
    -------
    pd.DataFrame
        Copy of mp with additional rolling mean/sum columns.
    """

    df = mp.copy()

    # Ensure datetime & sort order
    df["gameDate"] = pd.to_datetime(df["gameDate"], errors="coerce")
    df = df.sort_values(["team", "gameDate", "gameId"]).reset_index(drop=True)

    # Identify numeric columns to roll over
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # Exclude identifiers / things that don't make sense to roll
    exclude = {"season", "gameId"}   # you can add "playoffGame" here if you want
    numeric_cols = [c for c in numeric_cols if c not in exclude]

    print("Rolling over numeric columns:", len(numeric_cols))
    # print(numeric_cols)  # uncomment if you want to see them all

    grouped = df.groupby("team", group_keys=False)

    for col in numeric_cols:
        for w in windows:
            mean_col = f"{col}_roll{w}"
            sum_col  = f"{col}_sum{w}"

            # Rolling mean of previous w games
            df[mean_col] = grouped[col].apply(
                lambda s: s.shift(1).rolling(window=w, min_periods=min_games).mean()
            )

            # Rolling sum of previous w games
            df[sum_col] = grouped[col].apply(
                lambda s: s.shift(1).rolling(window=w, min_periods=min_games).sum()
            )

    return df


In [15]:
mp_with_roll = add_rolling_features_moneypuck(
    mp,
    windows=(1, 3, 5, 10),
    min_games=1
)

mp_with_roll.shape
mp_with_roll.filter(regex="xGoalsFor_roll|goalsFor_roll").head()


Rolling over numeric columns: 101


/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_63780/674495464.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[mean_col] = grouped[col].apply(
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_63780/674495464.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[sum_col] = grouped[col].apply(
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_63780/674495464.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has

,xGoalsFor_roll1,xGoalsFor_roll3,xGoalsFor_roll5,xGoalsFor_roll10,flurryAdjustedxGoalsFor_roll1,flurryAdjustedxGoalsFor_roll3,flurryAdjustedxGoalsFor_roll5,flurryAdjustedxGoalsFor_roll10,scoreVenueAdjustedxGoalsFor_roll1,scoreVenueAdjustedxGoalsFor_roll3,...,mediumDangerxGoalsFor_roll5,mediumDangerxGoalsFor_roll10,highDangerxGoalsFor_roll1,highDangerxGoalsFor_roll3,highDangerxGoalsFor_roll5,highDangerxGoalsFor_roll10,reboundxGoalsFor_roll1,reboundxGoalsFor_roll3,reboundxGoalsFor_roll5,reboundxGoalsFor_roll10
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.271,1.2710,1.27100,1.27100,1.234,1.234000,1.2340,1.2340,1.249,1.249000,...,0.622000,0.622000,0.000,0.000,0.00000,0.00000,0.230,0.230000,0.2300,0.2300
2,2.298,1.7845,1.78450,1.78450,2.277,1.755500,1.7555,1.7555,2.177,1.713000,...,0.858500,0.858500,0.000,0.000,0.00000,0.00000,0.000,0.115000,0.1150,0.1150
3,1.573,1.7140,1.71400,1.71400,1.541,1.684000,1.6840,1.6840,1.604,1.676667,...,0.760333,0.760333,0.201,0.067,0.06700,0.06700,0.148,0.126000,0.1260,0.1260
4,2.219,2.0300,1.84025,1.84025,2.192,2.003333,1.8110,1.8110,2.136,1.972333,...,0.796000,0.796000,0.000,0.067,0.05025,0.05025,0.000,0.049333,0.0945,0.0945


Merge datasets

In [50]:
import pandas as pd

# Load your master games file
base_games = pd.read_csv("nhl_games_2010_2024.csv")
base_games["game_date"] = pd.to_datetime(base_games["game_date"], errors="coerce")

# Map full team names -> 3-letter codes
TEAM_NAME_TO_CODE = {
    "Anaheim Ducks": "ANA",
    "Arizona Coyotes": "ARI",
    "Boston Bruins": "BOS",
    "Buffalo Sabres": "BUF",
    "Carolina Hurricanes": "CAR",
    "Columbus Blue Jackets": "CBJ",
    "Calgary Flames": "CGY",
    "Chicago Blackhawks": "CHI",
    "Colorado Avalanche": "COL",
    "Dallas Stars": "DAL",
    "Detroit Red Wings": "DET",
    "Edmonton Oilers": "EDM",
    "Florida Panthers": "FLA",
    "Los Angeles Kings": "LAK",
    "Minnesota Wild": "MIN",
    "Montreal Canadiens": "MTL",
    "Montréal Canadiens": "MTL",
    "New Jersey Devils": "NJD",
    "Nashville Predators": "NSH",
    "New York Islanders": "NYI",
    "New York Rangers": "NYR",
    "Ottawa Senators": "OTT",
    "Philadelphia Flyers": "PHI",
    "Pittsburgh Penguins": "PIT",
    "San Jose Sharks": "SJS",
    "St. Louis Blues": "STL",
    "Tampa Bay Lightning": "TBL",
    "Toronto Maple Leafs": "TOR",
    "Vancouver Canucks": "VAN",
    "Vegas Golden Knights": "VGK",
    "Washington Capitals": "WSH",
    "Winnipeg Jets": "WPG",
    # Historical names in your range
    "Phoenix Coyotes": "ARI",
    "Atlanta Thrashers": "ATL",
    # Expansion
    "Seattle Kraken": "SEA",
}

base_games["home_team_code"] = base_games["home_team"].map(TEAM_NAME_TO_CODE)
base_games["away_team_code"] = base_games["away_team"].map(TEAM_NAME_TO_CODE)

print("Missing home_team_code:", base_games["home_team_code"].isna().sum())
print("Missing away_team_code:", base_games["away_team_code"].isna().sum())

base_games.head()


Missing home_team_code: 0
Missing away_team_code: 0


,game_id,season,game_date,away_team,away_goals,home_team,home_goals,home_win,home_win_margin,home_team_code,away_team_code
0,20091001_WashingtonCapitals_@_BostonBruins,2010,2009-10-01,Washington Capitals,4,Boston Bruins,1,0,-3,BOS,WSH
1,20091001_VancouverCanucks_@_CalgaryFlames,2010,2009-10-01,Vancouver Canucks,3,Calgary Flames,5,1,2,CGY,VAN
2,20091001_SanJoseSharks_@_ColoradoAvalanche,2010,2009-10-01,San Jose Sharks,2,Colorado Avalanche,5,1,3,COL,SJS
3,20091001_MontrealCanadiens_@_TorontoMapleLeafs,2010,2009-10-01,Montreal Canadiens,4,Toronto Maple Leafs,3,0,-1,TOR,MTL
4,20091002_PhiladelphiaFlyers_@_CarolinaHurricanes,2010,2009-10-02,Philadelphia Flyers,2,Carolina Hurricanes,0,0,-2,CAR,PHI


In [52]:
import pandas as pd
import numpy as np

# 0) Make sure base_games & mp_with_roll are clean
base_games["game_date"] = pd.to_datetime(base_games["game_date"], errors="coerce")
mp_with_roll["gameDate"] = pd.to_datetime(mp_with_roll["gameDate"], errors="coerce")

# De-duplicate any weird duplicate columns just in case
base_games = base_games.loc[:, ~base_games.columns.duplicated()]
mp_with_roll = mp_with_roll.loc[:, ~mp_with_roll.columns.duplicated()]

# 1) Keep only what we need from MoneyPuck
rolling_cols = [c for c in mp_with_roll.columns if ("_roll" in c or "_sum" in c)]

mp_roll = mp_with_roll[["gameId", "gameDate", "team", "opposingTeam"] + rolling_cols].copy()
mp_roll = mp_roll.loc[:, ~mp_roll.columns.duplicated()]  # safety

print("mp_roll columns:", mp_roll.columns.tolist()[:15])

# 2) Build home-side rolling features (team = home, opposingTeam = away)
home_roll = mp_roll.copy()

# Prefix only the rolling columns with 'home_'
home_roll = home_roll.rename(columns={c: f"home_{c}" for c in rolling_cols})

print("home_roll sample columns:", home_roll.columns.tolist()[:15])

# 3) Build away-side rolling features (team = away, opposingTeam = home)
away_roll = mp_roll.copy()
away_roll = away_roll.rename(columns={c: f"away_{c}" for c in rolling_cols})

print("away_roll sample columns:", away_roll.columns.tolist()[:15])

# 4) Merge home features:
# match: base_games (game_date, home_team_code, away_team_code)
#    to: home_roll (gameDate, team, opposingTeam)

games_with_home_roll = base_games.merge(
    home_roll,
    left_on=["game_date", "home_team_code", "away_team_code"],
    right_on=["gameDate", "team", "opposingTeam"],
    how="left",
)

# Drop duplicate key columns from right
games_with_home_roll = games_with_home_roll.drop(columns=["gameDate", "team", "opposingTeam", "gameId"])

print("After home merge:", games_with_home_roll.shape)

# 5) Merge away features:
# match: base_games (game_date, home_team_code, away_team_code)
#    to: away_roll (gameDate, opposingTeam, team)  <-- swapped

games_with_full_roll = games_with_home_roll.merge(
    away_roll,
    left_on=["game_date", "home_team_code", "away_team_code"],
    right_on=["gameDate", "opposingTeam", "team"],
    how="left",
    suffixes=("", "_awaytmp")  # avoid conflicts if any
)

# Drop duplicate key columns from right
games_with_full_roll = games_with_full_roll.drop(columns=["gameDate", "team", "opposingTeam", "gameId"])

print("Final shape with MoneyPuck rolling features:", games_with_full_roll.shape)
games_with_full_roll.head()


mp_roll columns: ['gameId', 'gameDate', 'team', 'opposingTeam', 'xGoalsPercentage_roll1', 'xGoalsPercentage_sum1', 'xGoalsPercentage_roll3', 'xGoalsPercentage_sum3', 'xGoalsPercentage_roll5', 'xGoalsPercentage_sum5', 'xGoalsPercentage_roll10', 'xGoalsPercentage_sum10', 'corsiPercentage_roll1', 'corsiPercentage_sum1', 'corsiPercentage_roll3']
home_roll sample columns: ['gameId', 'gameDate', 'team', 'opposingTeam', 'home_xGoalsPercentage_roll1', 'home_xGoalsPercentage_sum1', 'home_xGoalsPercentage_roll3', 'home_xGoalsPercentage_sum3', 'home_xGoalsPercentage_roll5', 'home_xGoalsPercentage_sum5', 'home_xGoalsPercentage_roll10', 'home_xGoalsPercentage_sum10', 'home_corsiPercentage_roll1', 'home_corsiPercentage_sum1', 'home_corsiPercentage_roll3']
away_roll sample columns: ['gameId', 'gameDate', 'team', 'opposingTeam', 'away_xGoalsPercentage_roll1', 'away_xGoalsPercentage_sum1', 'away_xGoalsPercentage_roll3', 'away_xGoalsPercentage_sum3', 'away_xGoalsPercentage_roll5', 'away_xGoalsPercentage

,game_id,season,game_date,away_team,away_goals,home_team,home_goals,home_win,home_win_margin,home_team_code,...,away_scoreFlurryAdjustedTotalShotCreditAgainst_roll10,away_scoreFlurryAdjustedTotalShotCreditAgainst_sum10,away_playoffGame_roll1,away_playoffGame_sum1,away_playoffGame_roll3,away_playoffGame_sum3,away_playoffGame_roll5,away_playoffGame_sum5,away_playoffGame_roll10,away_playoffGame_sum10
0,20091001_WashingtonCapitals_@_BostonBruins,2010,2009-10-01,Washington Capitals,4,Boston Bruins,1,0,-3,BOS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20091001_VancouverCanucks_@_CalgaryFlames,2010,2009-10-01,Vancouver Canucks,3,Calgary Flames,5,1,2,CGY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20091001_SanJoseSharks_@_ColoradoAvalanche,2010,2009-10-01,San Jose Sharks,2,Colorado Avalanche,5,1,3,COL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20091001_MontrealCanadiens_@_TorontoMapleLeafs,2010,2009-10-01,Montreal Canadiens,4,Toronto Maple Leafs,3,0,-1,TOR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20091002_PhiladelphiaFlyers_@_CarolinaHurricanes,2010,2009-10-02,Philadelphia Flyers,2,Carolina Hurricanes,0,0,-2,CAR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
games_with_full_roll.to_csv('games_with_full_roll', index=False)

In [13]:
import pandas as pd
import numpy as np

# ================================================================
# 1) LOAD FILES
# ================================================================

games = pd.read_csv("games_with_full_roll", low_memory=False)
games["game_date"] = pd.to_datetime(games["game_date"])

odds_files = [
    "nhl_historical_odds_2020plus_pinnacle_betfair_dk_fd_ml_spreads_totals.csv",
]

odds_list = []
for f in odds_files:
    df = pd.read_csv(f, low_memory=False)
    odds_list.append(df)

odds_raw = pd.concat(odds_list, ignore_index=True)
odds = odds_raw.copy()

# ================================================================
# 2) CLEAN ODDS DATA
# ================================================================

# Parse commence_time in UTC
odds["commence_time"] = pd.to_datetime(
    odds["commence_time"], utc=True, errors="coerce"
)

# Convert to US/Eastern (approximate local game date) and drop timezone
odds["game_date"] = (
    odds["commence_time"]
    .dt.tz_convert("US/Eastern")
    .dt.normalize()
    .dt.tz_localize(None)
)

# Standardize column names
odds.rename(columns={
    "bookmaker": "book",
    "market": "market_type",
    "outcome_name": "team",
    "price": "odds"
}, inplace=True)

# Fix known team name mismatches
team_fix = {
    "St Louis Blues": "St. Louis Blues",
    "Montréal Canadiens": "Montreal Canadiens",
}
for col in ["home_team", "away_team", "team"]:
    odds[col] = odds[col].replace(team_fix)

# ================================================================
# 3) MONEYLINE (H2H)
# ================================================================

ml = odds[odds["market_type"] == "h2h"].copy()

# Drop 3-way "Draw" rows
ml = ml[(ml["team"] == ml["home_team"]) | (ml["team"] == ml["away_team"])]

ml["side"] = np.where(ml["team"] == ml["home_team"], "home", "away")

ml_pivot = ml.pivot_table(
    index=["game_date", "home_team", "away_team"],
    columns=["book", "side"],
    values="odds",
    aggfunc="last"
)

ml_pivot.columns = [f"{book}_ml_{side}" for (book, side) in ml_pivot.columns]
ml_pivot = ml_pivot.reset_index()

# ================================================================
# 4) SPREADS (POINT + PRICE)
# ================================================================

sp = odds[odds["market_type"] == "spreads"].copy()
sp = sp[(sp["team"] == sp["home_team"]) | (sp["team"] == sp["away_team"])]
sp["side"] = np.where(sp["team"] == sp["home_team"], "home", "away")

# Spread points
sp_pivot = sp.pivot_table(
    index=["game_date", "home_team", "away_team"],
    columns=["book", "side"],
    values="point",
    aggfunc="last"
)
sp_pivot.columns = [f"{book}_spread_{side}" for (book, side) in sp_pivot.columns]
sp_pivot = sp_pivot.reset_index()

# Spread odds
sp_price = sp.pivot_table(
    index=["game_date", "home_team", "away_team"],
    columns=["book", "side"],
    values="odds",
    aggfunc="last"
)
sp_price.columns = [f"{book}_spread_price_{side}" for (book, side) in sp_price.columns]
sp_price = sp_price.reset_index()

# ================================================================
# 5) TOTALS (POINT + PRICE)
# ================================================================

tot = odds[odds["market_type"] == "totals"].copy()

name_lower = tot["team"].astype(str).str.lower()
tot["OU"] = np.where(
    name_lower.str.contains("over"),
    "over",
    np.where(name_lower.str.contains("under"), "under", np.nan)
)
tot = tot[tot["OU"].notna()].copy()

# Total points
tot_pivot_point = tot.pivot_table(
    index=["game_date", "home_team", "away_team"],
    columns=["book", "OU"],
    values="point",
    aggfunc="last"
)
tot_pivot_point.columns = [
    f"{book}_total_{ou}" for (book, ou) in tot_pivot_point.columns
]
tot_pivot_point = tot_pivot_point.reset_index()

# Total odds
tot_pivot_price = tot.pivot_table(
    index=["game_date", "home_team", "away_team"],
    columns=["book", "OU"],
    values="odds",
    aggfunc="last"
)
tot_pivot_price.columns = [
    f"{book}_total_price_{ou}" for (book, ou) in tot_pivot_price.columns
]
tot_pivot_price = tot_pivot_price.reset_index()

# ================================================================
# 6) ENSURE DATE TYPES MATCH
# ================================================================

for df2 in [ml_pivot, sp_pivot, sp_price, tot_pivot_point, tot_pivot_price]:
    df2["game_date"] = pd.to_datetime(df2["game_date"]).dt.tz_localize(None)

# ================================================================
# 7) MERGE ALL ODDS ONTO GAMES
# ================================================================

df = games.merge(
    ml_pivot, on=["game_date", "home_team", "away_team"], how="left"
)
df = df.merge(
    sp_pivot, on=["game_date", "home_team", "away_team"], how="left"
)
df = df.merge(
    sp_price, on=["game_date", "home_team", "away_team"], how="left"
)
df = df.merge(
    tot_pivot_point, on=["game_date", "home_team", "away_team"], how="left"
)
df = df.merge(
    tot_pivot_price, on=["game_date", "home_team", "away_team"], how="left"
)

print("FINAL MERGED SHAPE:", df.shape)

# ================================================================
# 8) SAVE FINAL MERGED DATASET
# ================================================================

df.to_csv("games_with_full_roll_and_all_odds_cleaned.csv", index=False)
print("Saved to games_with_full_roll_and_all_odds_cleaned.csv")


FINAL MERGED SHAPE: (19118, 1657)
Saved to games_with_full_roll_and_all_odds_cleaned.csv


In [15]:
import pandas as pd

# Load your merged dataset
df = pd.read_csv("games_with_full_roll_and_all_odds_cleaned.csv", low_memory=False)

# -------------------------
# 1) Define required columns
# -------------------------

pinnacle_cols = [
    "pinnacle_ml_home", "pinnacle_ml_away",
    "pinnacle_spread_home", "pinnacle_spread_away",
    "pinnacle_total_over", "pinnacle_total_under"
]

fanduel_cols = [
    "fanduel_ml_home", "fanduel_ml_away",
    "fanduel_spread_home", "fanduel_spread_away",
    "fanduel_total_over", "fanduel_total_under"
]

required_cols = pinnacle_cols + fanduel_cols

# -------------------------
# 2) Create mask for rows with COMPLETE odds
# -------------------------

complete_mask = df[required_cols].notna().all(axis=1)

# -------------------------
# 3) Subset the dataset
# -------------------------

df_complete_pinnacle_dk = df[complete_mask].copy()

# Show summary
print("Rows with complete Pinnacle + DraftKings odds:", len(df_complete_pinnacle_dk))
df_complete_pinnacle_dk.head()


Rows with complete Pinnacle + DraftKings odds: 1556


,game_id,season,game_date,away_team,away_goals,home_team,home_goals,home_win,home_win_margin,home_team_code,...,fanduel_total_over,fanduel_total_under,pinnacle_total_over,pinnacle_total_under,draftkings_total_price_over,draftkings_total_price_under,fanduel_total_price_over,fanduel_total_price_under,pinnacle_total_price_over,pinnacle_total_price_under
14050,20210126_AnaheimDucks_@_ArizonaCoyotes,2021,2021-01-26,Anaheim Ducks,1,Arizona Coyotes,0,0,-1,ARI,...,5.5,5.5,5.5,5.5,NaN,NaN,110.0,-130.0,114.0,-128.0
14051,20210126_PittsburghPenguins_@_BostonBruins,2021,2021-01-26,Pittsburgh Penguins,2,Boston Bruins,3,1,1,BOS,...,5.5,5.5,5.5,5.5,-120.0,-103.0,-120.0,100.0,-114.0,101.0
14052,20210126_NewYorkRangers_@_BuffaloSabres,2021,2021-01-26,New York Rangers,2,Buffalo Sabres,3,1,1,BUF,...,6.5,6.5,6.0,6.0,NaN,NaN,105.0,-125.0,-110.0,-103.0
14053,20210126_FloridaPanthers_@_ColumbusBlueJackets,2021,2021-01-26,Florida Panthers,4,Columbus Blue Jackets,3,0,-1,CBJ,...,5.5,5.5,5.5,5.5,-112.0,-109.0,-110.0,-110.0,-102.0,-110.0
14054,20210126_TorontoMapleLeafs_@_CalgaryFlames,2021,2021-01-26,Toronto Maple Leafs,4,Calgary Flames,3,0,-1,CGY,...,6.5,6.5,6.0,6.0,-122.0,102.0,105.0,-120.0,-112.0,100.0


In [21]:
df_complete_pinnacle_dk.game_date

14050    2021-01-26
14051    2021-01-26
14052    2021-01-26
14053    2021-01-26
14054    2021-01-26
            ...    
15678    2022-02-15
15679    2022-02-16
15680    2022-02-16
15681    2022-02-16
15682    2022-02-16
Name: game_date, Length: 1556, dtype: object

In [23]:
import pandas as pd
import numpy as np

# ================================================================
# 1) LOAD DATA
# ================================================================

# Advanced stats + rolling features
games = pd.read_csv("games_with_full_roll", low_memory=False)
games["game_date"] = pd.to_datetime(games["game_date"])

# Combined odds from old + new scrapes
odds = pd.read_csv("odds.csv", low_memory=False)

print("games shape:", games.shape)
print("odds shape:", odds.shape)
print("odds columns:", odds.columns.tolist())


# ================================================================
# 2) STANDARDIZE ODDS COLUMNS
# ================================================================

# Some versions may have 'bookmaker', 'market', 'outcome_name', 'price'
rename_map = {}
if "bookmaker" in odds.columns:
    rename_map["bookmaker"] = "book"
if "market" in odds.columns:
    rename_map["market"] = "market_type"
if "outcome_name" in odds.columns:
    rename_map["outcome_name"] = "team"
if "price" in odds.columns:
    rename_map["price"] = "odds"

if rename_map:
    odds = odds.rename(columns=rename_map)

# Make sure required columns exist
required_cols = ["game_date", "home_team", "away_team", "book",
                 "market_type", "team", "odds"]
missing = [c for c in required_cols if c not in odds.columns]
if missing:
    raise ValueError(f"Missing required columns in odds: {missing}")

# Parse game_date as naive datetime
odds["game_date"] = pd.to_datetime(odds["game_date"], errors="coerce")

# Optional: ensure odds are numeric
odds["odds"] = pd.to_numeric(odds["odds"], errors="coerce")
if "point" in odds.columns:
    odds["point"] = pd.to_numeric(odds.get("point"), errors="coerce")


# ================================================================
# 3) FIX TEAM NAME MISMATCHES
# ================================================================

team_fix = {
    "St Louis Blues": "St. Louis Blues",
    "Montréal Canadiens": "Montreal Canadiens",
}
for col in ["home_team", "away_team", "team"]:
    if col in odds.columns:
        odds[col] = odds[col].replace(team_fix)


# ================================================================
# 4) MONEYLINE (H2H)
# ================================================================

ml = odds[odds["market_type"] == "h2h"].copy()

# Remove 3-way "Draw" (Pinnacle etc.)
ml = ml[(ml["team"] == ml["home_team"]) | (ml["team"] == ml["away_team"])]

ml["side"] = np.where(ml["team"] == ml["home_team"], "home", "away")

ml_pivot = ml.pivot_table(
    index=["game_date", "home_team", "away_team"],
    columns=["book", "side"],
    values="odds",
    aggfunc="last"
)

ml_pivot.columns = [f"{book}_ml_{side}" for (book, side) in ml_pivot.columns]
ml_pivot = ml_pivot.reset_index()

print("ML pivot shape:", ml_pivot.shape)


# ================================================================
# 5) SPREADS (POINTS + PRICES)
# ================================================================

sp = odds[odds["market_type"] == "spreads"].copy()
sp = sp[(sp["team"] == sp["home_team"]) | (sp["team"] == sp["away_team"])]
sp["side"] = np.where(sp["team"] == sp["home_team"], "home", "away")

# Spread points
sp_pivot = sp.pivot_table(
    index=["game_date", "home_team", "away_team"],
    columns=["book", "side"],
    values="point",
    aggfunc="last"
)
sp_pivot.columns = [f"{book}_spread_{side}" for (book, side) in sp_pivot.columns]
sp_pivot = sp_pivot.reset_index()

# Spread prices
sp_price = sp.pivot_table(
    index=["game_date", "home_team", "away_team"],
    columns=["book", "side"],
    values="odds",
    aggfunc="last"
)
sp_price.columns = [f"{book}_spread_price_{side}" for (book, side) in sp_price.columns]
sp_price = sp_price.reset_index()

print("Spread pivot shape:", sp_pivot.shape)
print("Spread price pivot shape:", sp_price.shape)


# ================================================================
# 6) TOTALS (POINTS + PRICES)
# ================================================================

tot = odds[odds["market_type"] == "totals"].copy()

name_lower = tot["team"].astype(str).str.lower()
tot["OU"] = np.where(
    name_lower.str.contains("over"),
    "over",
    np.where(name_lower.str.contains("under"), "under", np.nan)
)
tot = tot[tot["OU"].notna()].copy()

# Total points
tot_pivot_point = tot.pivot_table(
    index=["game_date", "home_team", "away_team"],
    columns=["book", "OU"],
    values="point",
    aggfunc="last"
)
tot_pivot_point.columns = [
    f"{book}_total_{ou}" for (book, ou) in tot_pivot_point.columns
]
tot_pivot_point = tot_pivot_point.reset_index()

# Total prices
tot_pivot_price = tot.pivot_table(
    index=["game_date", "home_team", "away_team"],
    columns=["book", "OU"],
    values="odds",
    aggfunc="last"
)
tot_pivot_price.columns = [
    f"{book}_total_price_{ou}" for (book, ou) in tot_pivot_price.columns
]
tot_pivot_price = tot_pivot_price.reset_index()

print("Totals pivot shape:", tot_pivot_point.shape)
print("Totals price pivot shape:", tot_pivot_price.shape)


# ================================================================
# 7) ENSURE DATE TYPES MATCH
# ================================================================

for df2 in [ml_pivot, sp_pivot, sp_price, tot_pivot_point, tot_pivot_price]:
    df2["game_date"] = pd.to_datetime(df2["game_date"]).dt.tz_localize(None)

games["game_date"] = pd.to_datetime(games["game_date"]).dt.tz_localize(None)


# ================================================================
# 8) MERGE ALL ODDS ONTO GAMES
# ================================================================

df = games.merge(
    ml_pivot, on=["game_date", "home_team", "away_team"], how="left"
)
df = df.merge(
    sp_pivot, on=["game_date", "home_team", "away_team"], how="left"
)
df = df.merge(
    sp_price, on=["game_date", "home_team", "away_team"], how="left"
)
df = df.merge(
    tot_pivot_point, on=["game_date", "home_team", "away_team"], how="left"
)
df = df.merge(
    tot_pivot_price, on=["game_date", "home_team", "away_team"], how="left"
)

print("FINAL MERGED SHAPE:", df.shape)

# Quick sanity check: show some odds columns
odds_cols = [c for c in df.columns if any(b in c for b in ["pinnacle", "fanduel", "draftkings", "betfair"])]
print("Sample odds columns:", odds_cols[:20])

# ================================================================
# 9) SAVE FINAL MERGED DATASET
# ================================================================

df.to_csv("games_with_full_roll_and_all_odds_cleaned.csv", index=False)
print("Saved to games_with_full_roll_and_all_odds_cleaned.csv")


games shape: (19118, 1627)
odds shape: (149790, 11)
odds columns: ['snapshot_ts', 'game_date', 'commence_time', 'event_id', 'home_team', 'away_team', 'book', 'market_type', 'team', 'odds', 'point']
ML pivot shape: (8396, 9)
Spread pivot shape: (8344, 9)
Spread price pivot shape: (8344, 9)
Totals pivot shape: (8388, 9)
Totals price pivot shape: (8388, 9)
FINAL MERGED SHAPE: (19118, 1657)
Sample odds columns: ['draftkings_ml_away', 'draftkings_ml_home', 'fanduel_ml_away', 'fanduel_ml_home', 'pinnacle_ml_away', 'pinnacle_ml_home', 'draftkings_spread_away', 'draftkings_spread_home', 'fanduel_spread_away', 'fanduel_spread_home', 'pinnacle_spread_away', 'pinnacle_spread_home', 'draftkings_spread_price_away', 'draftkings_spread_price_home', 'fanduel_spread_price_away', 'fanduel_spread_price_home', 'pinnacle_spread_price_away', 'pinnacle_spread_price_home', 'draftkings_total_over', 'draftkings_total_under']
Saved to games_with_full_roll_and_all_odds_cleaned.csv


In [27]:
df[df.season==2021].sample(20)

,game_id,season,game_date,away_team,away_goals,home_team,home_goals,home_win,home_win_margin,home_team_code,...,fanduel_total_over,fanduel_total_under,pinnacle_total_over,pinnacle_total_under,draftkings_total_price_over,draftkings_total_price_under,fanduel_total_price_over,fanduel_total_price_under,pinnacle_total_price_over,pinnacle_total_price_under
14073,20210128_LosAngelesKings_@_MinnesotaWild,2021,2021-01-28,Los Angeles Kings,3,Minnesota Wild,5,1,2,MIN,...,5.5,5.5,5.5,5.5,112.0,-135.0,110.0,-130.0,117.0,-132.0
14067,20210128_NewYorkRangers_@_BuffaloSabres,2021,2021-01-28,New York Rangers,3,Buffalo Sabres,2,0,-1,BUF,...,6.5,6.5,6.0,6.0,NaN,NaN,110.0,-130.0,-104.0,-108.0
14120,20210204_VancouverCanucks_@_TorontoMapleLeafs,2021,2021-02-04,Vancouver Canucks,3,Toronto Maple Leafs,7,1,4,TOR,...,6.5,6.5,6.5,6.5,-123.0,102.0,-130.0,110.0,-114.0,102.0
14307,20210304_BuffaloSabres_@_NewYorkIslanders,2021,2021-03-04,Buffalo Sabres,2,New York Islanders,5,1,3,NYI,...,5.5,5.5,5.5,5.5,117.0,-143.0,118.0,-145.0,122.0,-137.0
14674,20210422_NewJerseyDevils_@_PittsburghPenguins,2021,2021-04-22,New Jersey Devils,1,Pittsburgh Penguins,5,1,4,PIT,...,7.5,7.5,6.5,6.5,-149.0,118.0,116.0,-147.0,-143.0,113.0
14896,20210608_TampaBayLightning_@_CarolinaHurricanes,2021,2021-06-08,Tampa Bay Lightning,2,Carolina Hurricanes,0,0,-2,CAR,...,4.5,4.5,4.5,4.5,NaN,NaN,102.0,-130.0,-102.0,-123.0
14384,20210314_LosAngelesKings_@_ColoradoAvalanche,2021,2021-03-14,Los Angeles Kings,1,Colorado Avalanche,4,1,3,COL,...,2.5,2.5,5.5,5.5,170.0,-227.0,-1667.0,750.0,161.0,-204.0
14379,20210313_VegasGoldenKnights_@_St.LouisBlues,2021,2021-03-13,Vegas Golden Knights,5,St. Louis Blues,1,0,-4,STL,...,6.5,6.5,6.0,6.0,-114.0,-108.0,110.0,-133.0,-103.0,-109.0
14170,20210213_BostonBruins_@_NewYorkIslanders,2021,2021-02-13,Boston Bruins,2,New York Islanders,4,1,2,NYI,...,5.5,5.5,5.0,5.0,-127.0,105.0,135.0,-161.0,-120.0,107.0
14148,20210209_TampaBayLightning_@_NashvillePredators,2021,2021-02-09,Tampa Bay Lightning,6,Nashville Predators,1,0,-5,NSH,...,5.5,5.5,5.5,5.5,-120.0,-103.0,-120.0,100.0,-112.0,100.0


In [33]:
import pandas as pd

# === Load dataset ===
df = pd.read_csv("games_with_full_roll_and_all_odds_cleaned.csv")
print("Initial shape:", df.shape)

# ----------------------------------------------------------
# 1. Basic structural checks
# ----------------------------------------------------------

# Check duplicate game_ids
dupe_games = df[df.duplicated(subset=["game_id"], keep=False)]
print("Number of duplicated game_id rows:", dupe_games.shape[0])

# Optional: inspect a few
# print(dupe_games[["game_id", "game_date", "home_team", "away_team"]].head())

# Ensure game_date is a proper datetime
df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
invalid_dates = df["game_date"].isna().sum()
print("Invalid game_date entries:", invalid_dates)

# ----------------------------------------------------------
# 2. Identify Pinnacle odds columns (game-level)
# ----------------------------------------------------------
# This assumes your merged file has columns with "pinnacle" in the name,
# e.g. 'pinnacle_home_ml', 'pinnacle_away_ml', 'pinnacle_total', etc.

pinnacle_cols = [c for c in df.columns if "pinnacle" in c.lower()]

print("\nDetected Pinnacle-related columns:")
for c in pinnacle_cols:
    print("  -", c)

if not pinnacle_cols:
    raise ValueError("No Pinnacle columns detected. Check your column names or add 'pinnacle' to them.")

# ----------------------------------------------------------
# 3. Check Pinnacle completeness
# ----------------------------------------------------------
# "Complete Pinnacle data" = no NaNs in any Pinnacle column for that game_id

missing_pinnacle_mask = df[pinnacle_cols].isna().any(axis=1)
num_missing_games = missing_pinnacle_mask.sum()
total_games = df.shape[0]

print(f"\nGames with INCOMPLETE Pinnacle data: {num_missing_games} / {total_games}")

# Optional: inspect some problematic rows
# print(df.loc[missing_pinnacle_mask, ["game_id", "game_date", "home_team", "away_team"] + pinnacle_cols].head())

# ----------------------------------------------------------
# 4. Drop games without full Pinnacle odds
# ----------------------------------------------------------
clean_df = df.loc[~missing_pinnacle_mask].copy()
print("Shape AFTER dropping incomplete Pinnacle games:", clean_df.shape)

# Sanity check: no missing Pinnacle odds remain
print("Any remaining NaNs in Pinnacle cols:", clean_df[pinnacle_cols].isna().any().any())

# ----------------------------------------------------------
# 5. Save cleaned dataset
# ----------------------------------------------------------
output_path = "games_with_full_roll_and_all_odds_pinnacle_clean_only.csv"
clean_df.to_csv(output_path, index=False)
print(f"Saved cleaned file: {output_path}")

# ----------------------------------------------------------
# 6. Final integrity summary
# ----------------------------------------------------------
print("\nFinal integrity summary:")
print("Unique game_ids:", clean_df["game_id"].nunique())
print("Min date:", clean_df["game_date"].min())
print("Max date:", clean_df["game_date"].max())


Initial shape: (19118, 1657)
Number of duplicated game_id rows: 0
Invalid game_date entries: 0

Detected Pinnacle-related columns:
  - pinnacle_ml_away
  - pinnacle_ml_home
  - pinnacle_spread_away
  - pinnacle_spread_home
  - pinnacle_spread_price_away
  - pinnacle_spread_price_home
  - pinnacle_total_over
  - pinnacle_total_under
  - pinnacle_total_price_over
  - pinnacle_total_price_under

Games with INCOMPLETE Pinnacle data: 14351 / 19118
Shape AFTER dropping incomplete Pinnacle games: (4767, 1657)
Any remaining NaNs in Pinnacle cols: False
Saved cleaned file: games_with_full_roll_and_all_odds_pinnacle_clean_only.csv

Final integrity summary:
Unique game_ids: 4767
Min date: 2020-08-01 00:00:00
Max date: 2024-06-24 00:00:00


In [37]:
import pandas as pd
import numpy as np

# ============================
# 1. Load Pinnacle-clean dataset
# ============================
df = pd.read_csv("games_with_full_roll_and_all_odds_pinnacle_clean_only.csv")
print("Loaded:", df.shape)

# ============================
# 2. Utility functions
# ============================

def american_to_prob(odds):
    """
    Convert American odds to implied probability (with vig).
    odds: scalar or array-like
    """
    odds = np.asarray(odds, dtype=float)
    prob = np.where(
        odds > 0,
        100 / (odds + 100),
        np.where(odds < 0, -odds / (-odds + 100), np.nan)
    )
    return prob

def american_to_decimal(odds):
    """
    Convert American odds to decimal odds (including stake).
    """
    odds = np.asarray(odds, dtype=float)
    dec = np.where(
        odds > 0,
        1 + odds / 100,
        np.where(odds < 0, 1 + 100 / -odds, np.nan)
    )
    return dec

def fair_probs(p1, p2):
    """
    Remove vig by normalizing two implied probabilities to sum to 1.
    p1, p2: raw implied probabilities
    Returns: (p1_fair, p2_fair)
    """
    total = p1 + p2
    return p1 / total, p2 / total

# ============================
# 3. Moneyline implied probabilities (Pinnacle)
# ============================

# Raw implied probabilities with vig
df["pinnacle_ml_prob_home_raw"] = american_to_prob(df["pinnacle_ml_home"])
df["pinnacle_ml_prob_away_raw"] = american_to_prob(df["pinnacle_ml_away"])

# No-vig "fair" probabilities (normalized to sum to 1)
(df["pinnacle_ml_prob_home_fair"],
 df["pinnacle_ml_prob_away_fair"]) = fair_probs(
    df["pinnacle_ml_prob_home_raw"],
    df["pinnacle_ml_prob_away_raw"]
)

# Also store decimal odds (useful for EV calc)
df["pinnacle_ml_dec_home"] = american_to_decimal(df["pinnacle_ml_home"])
df["pinnacle_ml_dec_away"] = american_to_decimal(df["pinnacle_ml_away"])

# ============================
# 4. Spread implied probabilities (Pinnacle)
# ============================

# Spread prices are the odds for each side of the puckline
df["pinnacle_spread_prob_home_raw"] = american_to_prob(df["pinnacle_spread_price_home"])
df["pinnacle_spread_prob_away_raw"] = american_to_prob(df["pinnacle_spread_price_away"])

(df["pinnacle_spread_prob_home_fair"],
 df["pinnacle_spread_prob_away_fair"]) = fair_probs(
    df["pinnacle_spread_prob_home_raw"],
    df["pinnacle_spread_prob_away_raw"]
)

df["pinnacle_spread_dec_home"] = american_to_decimal(df["pinnacle_spread_price_home"])
df["pinnacle_spread_dec_away"] = american_to_decimal(df["pinnacle_spread_price_away"])

# Spread lines: usually mirror each other (e.g. home -1.5, away +1.5)
df["pinnacle_spread_home_line"] = df["pinnacle_spread_home"]
df["pinnacle_spread_away_line"] = df["pinnacle_spread_away"]

# ============================
# 5. Totals implied probabilities (Pinnacle)
# ============================

df["pinnacle_total_prob_over_raw"] = american_to_prob(df["pinnacle_total_price_over"])
df["pinnacle_total_prob_under_raw"] = american_to_prob(df["pinnacle_total_price_under"])

(df["pinnacle_total_prob_over_fair"],
 df["pinnacle_total_prob_under_fair"]) = fair_probs(
    df["pinnacle_total_prob_over_raw"],
    df["pinnacle_total_prob_under_raw"]
)

df["pinnacle_total_dec_over"] = american_to_decimal(df["pinnacle_total_price_over"])
df["pinnacle_total_dec_under"] = american_to_decimal(df["pinnacle_total_price_under"])

# Total line: over/under should share the same number; take the over column
df["pinnacle_total_line"] = df["pinnacle_total_over"]

# ============================
# 6. EV helper functions (run automatically when model preds exist)
# ============================

def ev_from_probs(model_prob, decimal_odds, stake=1.0):
    """
    Expected value of a bet:
    - model_prob: your model's probability of this outcome
    - decimal_odds: decimal odds for the bet (including stake)
    - stake: how much you risk per bet (default 1 unit)

    Profit if win = (decimal_odds - 1) * stake
    Profit if lose = -stake
    EV = p * profit_win + (1 - p) * profit_lose
    """
    profit_win = (decimal_odds - 1.0) * stake
    profit_lose = -stake
    return model_prob * profit_win + (1 - model_prob) * profit_lose

# We’ll only compute EV if model columns exist
cols = set(df.columns)

# Example: home/away moneyline EV if you later add:
# - 'model_home_win_prob' = model's probability home wins
# - 'model_away_win_prob' = model's probability away wins
if {"model_home_win_prob", "model_away_win_prob"} <= cols:
    df["ev_pinnacle_ml_home"] = ev_from_probs(
        df["model_home_win_prob"],
        df["pinnacle_ml_dec_home"]
    )
    df["ev_pinnacle_ml_away"] = ev_from_probs(
        df["model_away_win_prob"],
        df["pinnacle_ml_dec_away"]
    )

# Example: totals EV if you later add:
# - 'model_total_over_prob', 'model_total_under_prob'
if {"model_total_over_prob", "model_total_under_prob"} <= cols:
    df["ev_pinnacle_total_over"] = ev_from_probs(
        df["model_total_over_prob"],
        df["pinnacle_total_dec_over"]
    )
    df["ev_pinnacle_total_under"] = ev_from_probs(
        df["model_total_under_prob"],
        df["pinnacle_total_dec_under"]
    )

# Example: spreads EV if you later add:
# - 'model_spread_home_cover_prob', 'model_spread_away_cover_prob'
if {"model_spread_home_cover_prob", "model_spread_away_cover_prob"} <= cols:
    df["ev_pinnacle_spread_home"] = ev_from_probs(
        df["model_spread_home_cover_prob"],
        df["pinnacle_spread_dec_home"]
    )
    df["ev_pinnacle_spread_away"] = ev_from_probs(
        df["model_spread_away_cover_prob"],
        df["pinnacle_spread_dec_away"]
    )

# ============================
# 7. Save feature-enhanced dataset
# ============================
output_path = "games_with_full_roll_and_all_odds_pinnacle_features.csv"
df.to_csv(output_path, index=False)
print("Saved:", output_path)
print("Final shape:", df.shape)


Loaded: (4767, 1657)


/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/2924627145.py:23: RuntimeWarning: divide by zero encountered in divide
  np.where(odds < 0, -odds / (-odds + 100), np.nan)
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/2924627145.py:23: RuntimeWarning: divide by zero encountered in divide
  np.where(odds < 0, -odds / (-odds + 100), np.nan)
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/2924627145.py:23: RuntimeWarning: divide by zero encountered in divide
  np.where(odds < 0, -odds / (-odds + 100), np.nan)
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/2924627145.py:23: RuntimeWarning: divide by zero encountered in divide
  np.where(odds < 0, -odds / (-odds + 100), np.nan)
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/2924627145.py:23: RuntimeWarning: divide by zero encountered in divide
  np.where(odds < 0, -odds / (-odds + 100), np.nan)
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_9

Saved: games_with_full_roll_and_all_odds_pinnacle_features.csv
Final shape: (4767, 1678)


In [43]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Load games (NST features) and raw odds
# ---------------------------------------------------------
games = pd.read_csv("nhl_games_with_nst_season_features.csv")
odds = pd.read_csv("odds.csv")

print("Games shape:", games.shape)
print("Odds shape:", odds.shape)

# Ensure dates are datetime
games["game_date"] = pd.to_datetime(games["game_date"])
odds["game_date"] = pd.to_datetime(odds["game_date"])

# ---------------------------------------------------------
# 2. Clean team names in odds to match games
#    (extend mapping if you see more mismatches)
# ---------------------------------------------------------
team_name_map = {
    "Montréal Canadiens": "Montreal Canadiens",
    "St Louis Blues": "St. Louis Blues",
}
for col in ["home_team", "away_team", "team"]:
    if col in odds.columns:
        odds[col] = odds[col].replace(team_name_map)

# ---------------------------------------------------------
# 3. Attach game_id to each odds row by matching date+teams
# ---------------------------------------------------------
games_key = games[["game_id", "game_date", "home_team", "away_team"]]

odds_with_id = odds.merge(
    games_key,
    on=["game_date", "home_team", "away_team"],
    how="left",
    validate="m:1",  # many odds rows -> one game row
)

matched = odds_with_id[odds_with_id["game_id"].notna()].copy()
print("Matched odds rows:", matched.shape[0])
print("Games with at least one odds row:", matched["game_id"].nunique())

# ---------------------------------------------------------
# 4. Helper: from team -> home/away side
# ---------------------------------------------------------
def side_from_team(df):
    return np.where(
        df["team"] == df["home_team"], "home",
        np.where(df["team"] == df["away_team"], "away", df["team"])
    )

# ---------------------------------------------------------
# 5. Build wide odds for EACH book
# ---------------------------------------------------------
all_books = sorted(matched["book"].dropna().unique())
print("Books found:", all_books)

wide_tables = []

for book in all_books:
    book_key = str(book).lower().replace(" ", "")  # e.g. 'pinnacle', 'fanduel'
    sub = matched[matched["book"] == book].copy()
    print(f"\nProcessing book: {book}  (rows: {sub.shape[0]})")

    # Moneyline (h2h)
    ml = sub[sub["market_type"] == "h2h"].copy()
    ml["side"] = side_from_team(ml)
    ml = ml[ml["side"].isin(["home", "away"])]

    ml_pivot = ml.pivot_table(
        index="game_id",
        columns="side",
        values="odds",
        aggfunc="first",
    ).rename(
        columns={
            "home": f"{book_key}_ml_home",
            "away": f"{book_key}_ml_away",
        }
    )

    # Spreads
    sp = sub[sub["market_type"] == "spreads"].copy()
    if not sp.empty:
        sp["side"] = side_from_team(sp)
        sp = sp[sp["side"].isin(["home", "away"])]

        spread_point = sp.pivot_table(
            index="game_id",
            columns="side",
            values="point",
            aggfunc="first",
        ).rename(
            columns={
                "home": f"{book_key}_spread_home",
                "away": f"{book_key}_spread_away",
            }
        )

        spread_price = sp.pivot_table(
            index="game_id",
            columns="side",
            values="odds",
            aggfunc="first",
        ).rename(
            columns={
                "home": f"{book_key}_spread_price_home",
                "away": f"{book_key}_spread_price_away",
            }
        )
    else:
        spread_point = pd.DataFrame(index=ml_pivot.index)
        spread_price = pd.DataFrame(index=ml_pivot.index)

    # Totals
    tot = sub[sub["market_type"] == "totals"].copy()
    if not tot.empty:
        tot["side"] = tot["team"]  # 'Over' or 'Under'

        total_point = tot.pivot_table(
            index="game_id",
            columns="side",
            values="point",
            aggfunc="first",
        ).rename(
            columns={
                "Over": f"{book_key}_total_line_over",
                "Under": f"{book_key}_total_line_under",
            }
        )

        total_price = tot.pivot_table(
            index="game_id",
            columns="side",
            values="odds",
            aggfunc="first",
        ).rename(
            columns={
                "Over": f"{book_key}_total_price_over",
                "Under": f"{book_key}_total_price_under",
            }
        )

        # Single total line column (use Over if present, else Under)
        total_line = (
            total_point[[c for c in total_point.columns if "total_line" in c]]
            .bfill(axis=1)
            .iloc[:, 0]
            .to_frame(name=f"{book_key}_total_line")
        )
    else:
        total_point = pd.DataFrame(index=ml_pivot.index)
        total_price = pd.DataFrame(index=ml_pivot.index)
        total_line = pd.DataFrame(index=ml_pivot.index)

    # Combine this book's info into one wide table
    book_wide = (
        ml_pivot
        .join(spread_point, how="outer")
        .join(spread_price, how="outer")
        .join(total_point, how="outer")
        .join(total_price, how="outer")
        .join(total_line, how="outer")
    )

    wide_tables.append(book_wide)

# ---------------------------------------------------------
# 6. Join all books' wide tables on game_id
# ---------------------------------------------------------
if wide_tables:
    odds_wide_all = wide_tables[0]
    for tbl in wide_tables[1:]:
        odds_wide_all = odds_wide_all.join(tbl, how="outer")

    print("\nCombined wide odds shape:", odds_wide_all.shape)
else:
    raise ValueError("No odds tables were built; check odds.csv contents.")

# ---------------------------------------------------------
# 7. Merge wide odds onto the games dataset
# ---------------------------------------------------------
games_with_all_odds = games.merge(
    odds_wide_all,
    left_on="game_id",
    right_index=True,
    how="left",
)

print("Final merged shape (games + all books):", games_with_all_odds.shape)

for book in all_books:
    bk = str(book).lower().replace(" ", "")
    col = f"{bk}_ml_home"
    if col in games_with_all_odds.columns:
        print(f"Rows with {book} ML odds:", games_with_all_odds[col].notna().sum())

# ---------------------------------------------------------
# 8. Save final dataset
# ---------------------------------------------------------
out_path = "nhl_games_with_nst_and_all_odds.csv"
games_with_all_odds.to_csv(out_path, index=False)
print(f"Saved merged dataset to: {out_path}")


Games shape: (19118, 95)
Odds shape: (149790, 11)
Matched odds rows: 99452
Games with at least one odds row: 4936
Books found: ['draftkings', 'fanduel', 'pinnacle']

Processing book: draftkings  (rows: 33282)

Processing book: fanduel  (rows: 30248)

Processing book: pinnacle  (rows: 35922)

Combined wide odds shape: (4936, 33)
Final merged shape (games + all books): (19118, 128)
Rows with draftkings ML odds: 4788
Rows with fanduel ML odds: 4851
Rows with pinnacle ML odds: 4770
Saved merged dataset to: nhl_games_with_nst_and_all_odds.csv


In [45]:
import pandas as pd
import numpy as np

# ============================
# 1. Load merged games + odds
# ============================
df = pd.read_csv("nhl_games_with_nst_and_all_odds.csv")
print("Loaded:", df.shape)

# ============================
# 2. Utility functions
# ============================

def american_to_prob(odds):
    """
    Convert American odds to implied probability (with vig).
    odds: scalar or array-like; returns np.array of probabilities
    """
    odds = np.asarray(odds, dtype=float)
    prob = np.where(
        np.isnan(odds),
        np.nan,
        np.where(
            odds > 0,
            100 / (odds + 100),
            np.where(
                odds < 0,
                -odds / (-odds + 100),
                np.nan
            ),
        ),
    )
    return prob

def american_to_decimal(odds):
    """
    Convert American odds to decimal odds (including stake).
    """
    odds = np.asarray(odds, dtype=float)
    dec = np.where(
        np.isnan(odds),
        np.nan,
        np.where(
            odds > 0,
            1 + odds / 100,
            np.where(
                odds < 0,
                1 + 100 / -odds,
                np.nan
            ),
        ),
    )
    return dec

def fair_probs(p1, p2):
    """
    Remove vig by normalizing two implied probabilities to sum to 1.
    p1, p2: arrays of raw implied probabilities
    Returns: (p1_fair, p2_fair)
    """
    total = p1 + p2
    # avoid division by zero
    with np.errstate(divide="ignore", invalid="ignore"):
        p1_fair = np.where(total > 0, p1 / total, np.nan)
        p2_fair = np.where(total > 0, p2 / total, np.nan)
    return p1_fair, p2_fair

def ev_from_probs(model_prob, decimal_odds, stake=1.0):
    """
    Expected value of a bet:
    - model_prob: model's probability of this outcome
    - decimal_odds: decimal odds for the bet (including stake)
    - stake: amount risked per bet (default 1 unit)

    Profit if win  = (decimal_odds - 1) * stake
    Profit if lose = -stake
    EV = p * profit_win + (1 - p) * profit_lose
    """
    model_prob = np.asarray(model_prob, dtype=float)
    decimal_odds = np.asarray(decimal_odds, dtype=float)

    profit_win = (decimal_odds - 1.0) * stake
    profit_lose = -stake
    ev = model_prob * profit_win + (1 - model_prob) * profit_lose
    return ev

# ============================
# 3. Detect books from columns
# ============================

# any column ending with '_ml_home' is a book
ml_home_cols = [c for c in df.columns if c.endswith("_ml_home")]
books = [c.replace("_ml_home", "") for c in ml_home_cols]

print("Detected books:", books)

# ============================
# 4. Feature engineering per book
# ============================
for book in books:
    print(f"\nProcessing book: {book}")

    # ---------- Moneyline ----------
    ml_home_col = f"{book}_ml_home"
    ml_away_col = f"{book}_ml_away"

    if ml_home_col in df.columns and ml_away_col in df.columns:
        # raw implied probabilities
        home_raw = american_to_prob(df[ml_home_col])
        away_raw = american_to_prob(df[ml_away_col])

        df[f"{book}_ml_prob_home_raw"] = home_raw
        df[f"{book}_ml_prob_away_raw"] = away_raw

        # no-vig fair probabilities
        home_fair, away_fair = fair_probs(home_raw, away_raw)
        df[f"{book}_ml_prob_home_fair"] = home_fair
        df[f"{book}_ml_prob_away_fair"] = away_fair

        # decimal odds
        df[f"{book}_ml_dec_home"] = american_to_decimal(df[ml_home_col])
        df[f"{book}_ml_dec_away"] = american_to_decimal(df[ml_away_col])

    # ---------- Spreads ----------
    spread_price_home_col = f"{book}_spread_price_home"
    spread_price_away_col = f"{book}_spread_price_away"
    spread_home_col = f"{book}_spread_home"
    spread_away_col = f"{book}_spread_away"

    if spread_price_home_col in df.columns and spread_price_away_col in df.columns:
        sp_home_raw = american_to_prob(df[spread_price_home_col])
        sp_away_raw = american_to_prob(df[spread_price_away_col])

        df[f"{book}_spread_prob_home_raw"] = sp_home_raw
        df[f"{book}_spread_prob_away_raw"] = sp_away_raw

        sp_home_fair, sp_away_fair = fair_probs(sp_home_raw, sp_away_raw)
        df[f"{book}_spread_prob_home_fair"] = sp_home_fair
        df[f"{book}_spread_prob_away_fair"] = sp_away_fair

        df[f"{book}_spread_dec_home"] = american_to_decimal(df[spread_price_home_col])
        df[f"{book}_spread_dec_away"] = american_to_decimal(df[spread_price_away_col])

    # expose spread lines directly (if not already:
    if spread_home_col in df.columns:
        df[f"{book}_spread_home_line"] = df[spread_home_col]
    if spread_away_col in df.columns:
        df[f"{book}_spread_away_line"] = df[spread_away_col]

    # ---------- Totals ----------
    total_price_over_col = f"{book}_total_price_over"
    total_price_under_col = f"{book}_total_price_under"
    total_line_col = f"{book}_total_line"

    if total_price_over_col in df.columns and total_price_under_col in df.columns:
        tot_over_raw = american_to_prob(df[total_price_over_col])
        tot_under_raw = american_to_prob(df[total_price_under_col])

        df[f"{book}_total_prob_over_raw"] = tot_over_raw
        df[f"{book}_total_prob_under_raw"] = tot_under_raw

        tot_over_fair, tot_under_fair = fair_probs(tot_over_raw, tot_under_raw)
        df[f"{book}_total_prob_over_fair"] = tot_over_fair
        df[f"{book}_total_prob_under_fair"] = tot_under_fair

        df[f"{book}_total_dec_over"] = american_to_decimal(df[total_price_over_col])
        df[f"{book}_total_dec_under"] = american_to_decimal(df[total_price_under_col])

    # ensure single total line column exists
    if total_line_col in df.columns:
        # already there from merge step; keep as main total line
        pass

    # ---------- EV helpers (conditional) ----------
    # These only compute if you already have model prediction columns.
    # Example model columns:
    #   model_home_win_prob
    #   model_away_win_prob
    #   model_total_over_prob
    #   model_total_under_prob
    #   model_spread_home_cover_prob
    #   model_spread_away_cover_prob

    cols = set(df.columns)

    # Moneyline EV
    if {"model_home_win_prob", "model_away_win_prob"} <= cols and \
       {f"{book}_ml_dec_home", f"{book}_ml_dec_away"} <= cols:

        df[f"{book}_ev_ml_home"] = ev_from_probs(
            df["model_home_win_prob"],
            df[f"{book}_ml_dec_home"]
        )
        df[f"{book}_ev_ml_away"] = ev_from_probs(
            df["model_away_win_prob"],
            df[f"{book}_ml_dec_away"]
        )

    # Totals EV
    if {"model_total_over_prob", "model_total_under_prob"} <= cols and \
       {f"{book}_total_dec_over", f"{book}_total_dec_under"} <= cols:

        df[f"{book}_ev_total_over"] = ev_from_probs(
            df["model_total_over_prob"],
            df[f"{book}_total_dec_over"]
        )
        df[f"{book}_ev_total_under"] = ev_from_probs(
            df["model_total_under_prob"],
            df[f"{book}_total_dec_under"]
        )

    # Spread EV
    if {"model_spread_home_cover_prob", "model_spread_away_cover_prob"} <= cols and \
       {f"{book}_spread_dec_home", f"{book}_spread_dec_away"} <= cols:

        df[f"{book}_ev_spread_home"] = ev_from_probs(
            df["model_spread_home_cover_prob"],
            df[f"{book}_spread_dec_home"]
        )
        df[f"{book}_ev_spread_away"] = ev_from_probs(
            df["model_spread_away_cover_prob"],
            df[f"{book}_spread_dec_away"]
        )

# ============================
# 5. Save feature-enhanced dataset
# ============================
out_path = "nhl_games_with_nst_and_all_odds_features.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)
print("Final shape:", df.shape)


Loaded: (19118, 128)
Detected books: ['draftkings', 'fanduel', 'pinnacle']

Processing book: draftkings

Processing book: fanduel

Processing book: pinnacle


/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/3649570339.py:28: RuntimeWarning: divide by zero encountered in divide
  -odds / (-odds + 100),
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/3649570339.py:28: RuntimeWarning: divide by zero encountered in divide
  -odds / (-odds + 100),
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/3649570339.py:28: RuntimeWarning: divide by zero encountered in divide
  -odds / (-odds + 100),
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/3649570339.py:28: RuntimeWarning: divide by zero encountered in divide
  -odds / (-odds + 100),
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/3649570339.py:28: RuntimeWarning: divide by zero encountered in divide
  -odds / (-odds + 100),
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/3649570339.py:28: RuntimeWarning: divide by zero encountered in divide
  -odds / (-odds + 100),
/var/folders/fz/0282wvb93rn0lm_p0n

Saved: nhl_games_with_nst_and_all_odds_features.csv
Final shape: (19118, 188)


In [47]:
import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD BASE DATA (NST + ODDS + IMPLIED PROBS)
# ============================================================

base_path = "nhl_games_with_nst_and_all_odds_features.csv"
df = pd.read_csv(base_path)

print("Loaded:", df.shape)

# Ensure dates are datetime
df["game_date"] = pd.to_datetime(df["game_date"])

# ============================================================
# 2. BUILD LONG FORMAT (ONE ROW PER TEAM PER GAME)
#    This lets us compute rolling stats across home + away games
# ============================================================

# Columns we know exist from your screenshot
base_stats = [
    "GP", "TOI",
    "CF", "CA", "CFPct",
    "FF", "FA", "FFPct",
    "SF", "SA", "SFPct",
    "GF", "GA", "GFPct",
    "xGF", "xGA", "xGFPct",
    "SCF", "SCA", "SCFPct",
    "SCSF", "SCSA", "SCSFPct",
    "SCGF", "SCGA", "SCGFPct",
    "HDCF", "HDCA", "HDCFPct",
    "HDSF", "HDSA", "HDSFPct",
    "HDGF", "HDGA", "HDGFPct",
    "SHPct", "SVPct", "PDO",
]

# Keep only the ones that definitely exist
home_cols = [f"home_{c}" for c in base_stats if f"home_{c}" in df.columns]
away_cols = [f"away_{c}" for c in base_stats if f"away_{c}" in df.columns]

print("Home stat cols:", len(home_cols))
print("Away stat cols:", len(away_cols))

# Optional: use season if it exists for grouping
has_season = "season" in df.columns

# Build long/home
home_long = df[["game_id", "game_date", "home_team"] + (["season"] if has_season else []) + home_cols].copy()
home_long = home_long.rename(columns={"home_team": "team"})
home_long["is_home"] = 1

# Strip "home_" prefix to make generic stat names
home_long = home_long.rename(
    columns={c: c.replace("home_", "") for c in home_cols}
)

# Build long/away
away_long = df[["game_id", "game_date", "away_team"] + (["season"] if has_season else []) + away_cols].copy()
away_long = away_long.rename(columns={"away_team": "team"})
away_long["is_home"] = 0
away_long = away_long.rename(
    columns={c: c.replace("away_", "") for c in away_cols}
)

# Ensure we use the same stat set for home/away
common_stats = sorted(set(home_long.columns) & set(away_long.columns))
# common_stats includes: game_id, game_date, team, is_home, [season?], and shared stats

long_df = pd.concat([home_long[common_stats], away_long[common_stats]], ignore_index=True)
print("Long format shape (team-games):", long_df.shape)

# ============================================================
# 3. DEFINE ROLLING / EWMA / TREND CONFIG
# ============================================================

windows = [3, 5, 10, 20]       # multi-horizon rolling
ewma_alphas = [0.2, 0.05]      # short and medium EWMA

# Key stats for form/trend
form_stats = [
    "xGFPct", "CFPct", "HDCFPct", "GFPct", "PDO"
]

# For rolling means, we’ll use both percentages + core counts
rolling_stats = [
    "CFPct", "FFPct", "SFPct", "GFPct", "xGFPct",
    "SCFPct", "SCSFPct", "SCGFPct",
    "HDCFPct", "HDSFPct", "HDGFPct",
    "PDO",
    "CF", "CA", "SF", "SA", "GF", "GA",
    "xGF", "xGA", "HDCF", "HDCA", "HDGF", "HDGA",
]

# Keep only those actually present in long_df
rolling_stats = [s for s in rolling_stats if s in long_df.columns]
form_stats = [s for s in form_stats if s in long_df.columns]

print("Rolling stats:", rolling_stats)
print("Form/trend stats:", form_stats)

# Grouping keys (team + optional season)
group_keys = ["team"] + (["season"] if has_season else [])

# Sort for consistent rolling behavior
long_df = long_df.sort_values(group_keys + ["game_date"])

# ============================================================
# 4. COMPUTE ROLLING WINDOWS & EWMA
# ============================================================

for stat in rolling_stats:
    print(f"Computing rolling + EWMA for {stat}")
    grp = long_df.groupby(group_keys)[stat]

    # Shift by 1 so only PAST games are used (no leakage)
    shifted = grp.shift(1)

    # Simple rolling means
    for w in windows:
        col_name = f"{stat}_roll{w}"
        long_df[col_name] = shifted.groupby(long_df["team"]).rolling(
            window=w, min_periods=1
        ).mean().reset_index(level=0, drop=True)

    # EWMA
    for alpha in ewma_alphas:
        col_name = f"{stat}_ewm{str(alpha).replace('.', '')}"
        long_df[col_name] = shifted.groupby(long_df["team"]).ewm(
            alpha=alpha, adjust=False
        ).mean().reset_index(level=0, drop=True)

# ============================================================
# 5. GOAL DIFF + WIN RATE ROLLING
# ============================================================

if {"GF", "GA"} <= set(long_df.columns):
    long_df["goal_diff"] = long_df["GF"] - long_df["GA"]
    long_df["win_flag"] = (long_df["GF"] > long_df["GA"]).astype(int)

    grp_goal = long_df.groupby(group_keys)["goal_diff"]
    grp_win = long_df.groupby(group_keys)["win_flag"]

    long_df["goal_diff_roll5"] = grp_goal.shift(1).rolling(5, min_periods=1).mean()
    long_df["goal_diff_roll10"] = grp_goal.shift(1).rolling(10, min_periods=1).mean()
    long_df["goal_diff_roll20"] = grp_goal.shift(1).rolling(20, min_periods=1).mean()

    long_df["win_pct_roll10"] = grp_win.shift(1).rolling(10, min_periods=1).mean()
else:
    print("Warning: GF/GA not both present; skipping goal_diff and win_pct.")

# ============================================================
# 6. TREND FEATURES (SHORT VS LONG FORM)
# ============================================================

for stat in form_stats:
    # 5 vs 20 games
    s5 = f"{stat}_roll5"
    s20 = f"{stat}_roll20"
    if s5 in long_df.columns and s20 in long_df.columns:
        long_df[f"{stat}_trend_5v20"] = long_df[s5] - long_df[s20]

    # 3 vs 10 games
    s3 = f"{stat}_roll3"
    s10 = f"{stat}_roll10"
    if s3 in long_df.columns and s10 in long_df.columns:
        long_df[f"{stat}_trend_3v10"] = long_df[s3] - long_df[s10]

# ============================================================
# 7. PREP FOR MERGE BACK TO WIDE (HOME/AWAY)
# ============================================================

# We only need rolling / ewma / trend columns + goal_diff/win_pct
feature_cols = [
    c for c in long_df.columns
    if any(
        tag in c
        for tag in [
            "_roll3", "_roll5", "_roll10", "_roll20",
            "_ewm", "trend_5v20", "trend_3v10",
            "goal_diff", "win_pct_roll10"
        ]
    )
]

# Also keep game_id and is_home to join back
feature_cols = ["game_id", "is_home"] + feature_cols

features_long = long_df[feature_cols].copy()

# Split into home / away
home_features_long = features_long[features_long["is_home"] == 1].drop(columns=["is_home"])
away_features_long = features_long[features_long["is_home"] == 0].drop(columns=["is_home"])

# Rename generic stat names to home_* and away_*
home_features_long = home_features_long.rename(
    columns={c: f"home_{c}" for c in home_features_long.columns if c not in ["game_id"]}
)
away_features_long = away_features_long.rename(
    columns={c: f"away_{c}" for c in away_features_long.columns if c not in ["game_id"]}
)

print("Home rolling feature shape:", home_features_long.shape)
print("Away rolling feature shape:", away_features_long.shape)

# ============================================================
# 8. MERGE ROLLING FEATURES BACK INTO ORIGINAL WIDE DF
# ============================================================

df_merged = df.merge(home_features_long, on="game_id", how="left")
df_merged = df_merged.merge(away_features_long, on="game_id", how="left")

print("Merged with rolling features:", df_merged.shape)

# ============================================================
# 9. OPPONENT-ADJUSTED FEATURES (HOME - AWAY DIFFERENCES)
#    These are super useful & interpretable.
# ============================================================

# Choose a core set of rolling stats to differentialize
core_for_diff = [
    "xGFPct_roll10",
    "CFPct_roll10",
    "HDCFPct_roll10",
    "GFPct_roll10",
    "PDO_roll10",
    "goal_diff_roll10",
    "win_pct_roll10",
]

for base in core_for_diff:
    home_col = f"home_{base}"
    away_col = f"away_{base}"
    if home_col in df_merged.columns and away_col in df_merged.columns:
        diff_col = f"diff_{base}"
        df_merged[diff_col] = df_merged[home_col] - df_merged[away_col]

# ============================================================
# 10. SAVE FINAL DATASET
# ============================================================

out_path = "nhl_games_with_nst_all_odds_rolling_features.csv"
df_merged.to_csv(out_path, index=False)

print("Saved full feature dataset to:", out_path)


Loaded: (19118, 188)
Home stat cols: 38
Away stat cols: 38
Long format shape (team-games): (38236, 43)
Rolling stats: ['CFPct', 'FFPct', 'SFPct', 'GFPct', 'xGFPct', 'SCFPct', 'SCSFPct', 'SCGFPct', 'HDCFPct', 'HDSFPct', 'HDGFPct', 'PDO', 'CF', 'CA', 'SF', 'SA', 'GF', 'GA', 'xGF', 'xGA', 'HDCF', 'HDCA', 'HDGF', 'HDGA']
Form/trend stats: ['xGFPct', 'CFPct', 'HDCFPct', 'GFPct', 'PDO']
Computing rolling + EWMA for CFPct
Computing rolling + EWMA for FFPct
Computing rolling + EWMA for SFPct
Computing rolling + EWMA for GFPct
Computing rolling + EWMA for xGFPct
Computing rolling + EWMA for SCFPct
Computing rolling + EWMA for SCSFPct
Computing rolling + EWMA for SCGFPct
Computing rolling + EWMA for HDCFPct
Computing rolling + EWMA for HDSFPct
Computing rolling + EWMA for HDGFPct
Computing rolling + EWMA for PDO
Computing rolling + EWMA for CF
Computing rolling + EWMA for CA
Computing rolling + EWMA for SF
Computing rolling + EWMA for SA
Computing rolling + EWMA for GF
Computing rolling + EWMA f

/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/2303447254.py:129: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  long_df[col_name] = shifted.groupby(long_df["team"]).ewm(
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/2303447254.py:129: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  long_df[col_name] = shifted.groupby(long_df["team"]).ewm(
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/2303447254.py:122: PerformanceWarning: DataFrame is highly fragmented.  This is usually the r

Computing rolling + EWMA for HDCF
Computing rolling + EWMA for HDCA
Computing rolling + EWMA for HDGF
Computing rolling + EWMA for HDGA
Home rolling feature shape: (19118, 160)
Away rolling feature shape: (19118, 160)
Merged with rolling features: (19118, 506)


/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/2303447254.py:122: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  long_df[col_name] = shifted.groupby(long_df["team"]).rolling(
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/2303447254.py:122: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  long_df[col_name] = shifted.groupby(long_df["team"]).rolling(
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/2303447254.py:122: PerformanceWarning: DataFrame is highly fragmented.  This is usual

Saved full feature dataset to: nhl_games_with_nst_all_odds_rolling_features.csv


In [49]:
df = pd.read_csv("nhl_games_with_nst_all_odds_rolling_features.csv")
df = df.sort_values("game_date")
df.isna().mean().sort_values(ascending=False).head(30)


draftkings_spread_prob_away_fair    0.807616
draftkings_spread_home_line         0.807616
draftkings_spread_home              0.807616
draftkings_spread_price_away        0.807616
draftkings_spread_price_home        0.807616
draftkings_spread_prob_away_raw     0.807616
draftkings_spread_prob_home_fair    0.807616
draftkings_spread_dec_home          0.807616
draftkings_spread_dec_away          0.807616
draftkings_spread_away_line         0.807616
draftkings_spread_prob_home_raw     0.807616
draftkings_spread_away              0.807616
fanduel_spread_prob_home_raw        0.801653
fanduel_spread_prob_away_raw        0.801653
fanduel_spread_prob_home_fair       0.801653
fanduel_spread_prob_away_fair       0.801653
fanduel_spread_dec_home             0.801653
fanduel_spread_dec_away             0.801653
fanduel_spread_home_line            0.801653
fanduel_spread_away_line            0.801653
fanduel_spread_price_home           0.801653
fanduel_spread_price_away           0.801653
fanduel_sp

In [51]:
df.head()

,game_id,season,game_date,away_team,away_goals,home_team,home_goals,home_win,home_win_margin,home_team_code,...,away_GFPct_trend_3v10,away_PDO_trend_5v20,away_PDO_trend_3v10,diff_xGFPct_roll10,diff_CFPct_roll10,diff_HDCFPct_roll10,diff_GFPct_roll10,diff_PDO_roll10,diff_goal_diff_roll10,diff_win_pct_roll10
0,20091001_WashingtonCapitals_@_BostonBruins,2010,2009-10-01,Washington Capitals,4,Boston Bruins,1,0,-3,BOS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0
1,20091001_VancouverCanucks_@_CalgaryFlames,2010,2009-10-01,Vancouver Canucks,3,Calgary Flames,5,1,2,CGY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-60.0,-1.0
2,20091001_SanJoseSharks_@_ColoradoAvalanche,2010,2009-10-01,San Jose Sharks,2,Colorado Avalanche,5,1,3,COL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-23.0,0.0
3,20091001_MontrealCanadiens_@_TorontoMapleLeafs,2010,2009-10-01,Montreal Canadiens,4,Toronto Maple Leafs,3,0,-1,TOR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,20091002_PhiladelphiaFlyers_@_CarolinaHurricanes,2010,2009-10-02,Philadelphia Flyers,2,Carolina Hurricanes,0,0,-2,CAR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-26.0,-1.0


In [61]:
print(list(df.columns))

['game_id', 'season', 'game_date', 'away_team', 'away_goals', 'home_team', 'home_goals', 'home_win', 'home_win_margin', 'home_team_code', 'away_team_code', 'home_GP', 'home_TOI', 'home_CF', 'home_CA', 'home_CFPct', 'home_FF', 'home_FA', 'home_FFPct', 'home_SF', 'home_SA', 'home_SFPct', 'home_GF', 'home_GA', 'home_GFPct', 'home_xGF', 'home_xGA', 'home_xGFPct', 'home_SCF', 'home_SCA', 'home_SCFPct', 'home_SCSF', 'home_SCSA', 'home_SCSFPct', 'home_SCGF', 'home_SCGA', 'home_SCGFPct', 'home_SCSHPct', 'home_SCSVPct', 'home_HDCF', 'home_HDCA', 'home_HDCFPct', 'home_HDSF', 'home_HDSA', 'home_HDSFPct', 'home_HDGF', 'home_HDGA', 'home_HDGFPct', 'home_HDSHPct', 'home_HDSVPct', 'home_SHPct', 'home_SVPct', 'home_PDO', 'away_GP', 'away_TOI', 'away_CF', 'away_CA', 'away_CFPct', 'away_FF', 'away_FA', 'away_FFPct', 'away_SF', 'away_SA', 'away_SFPct', 'away_GF', 'away_GA', 'away_GFPct', 'away_xGF', 'away_xGA', 'away_xGFPct', 'away_SCF', 'away_SCA', 'away_SCFPct', 'away_SCSF', 'away_SCSA', 'away_SCSFPct'

In [63]:
df[['season','game_date','home_team','pinnacle_ml_home']]

,season,game_date,home_team,pinnacle_ml_home
0,2010,2009-10-01,Boston Bruins,NaN
1,2010,2009-10-01,Calgary Flames,NaN
2,2010,2009-10-01,Colorado Avalanche,NaN
3,2010,2009-10-01,Toronto Maple Leafs,NaN
4,2010,2009-10-02,Carolina Hurricanes,NaN
...,...,...,...,...
19113,2024,2024-06-13,Edmonton Oilers,-139.0
19114,2024,2024-06-15,Edmonton Oilers,-117.0
19115,2024,2024-06-18,Florida Panthers,-149.0
19116,2024,2024-06-21,Edmonton Oilers,-115.0


In [97]:
import pandas as pd

# Load your odds file
odds = pd.read_csv("odds.csv")

# Ensure snapshot_ts is datetime (critical)
odds['snapshot_ts'] = pd.to_datetime(odds['snapshot_ts'], utc=True, errors='coerce')

# Sort so the LAST row per group is kept
odds = odds.sort_values(['snapshot_ts'])

# Define the columns that uniquely identify a line
dedupe_cols = [
    'game_date',
    'home_team',
    'away_team',
    'market_type',
    'book',
    'team',  # or outcome_name depending on file
]

# Some files use different naming structure — adjust if needed
dedupe_cols = [c for c in dedupe_cols if c in odds.columns]

# Drop duplicates, keeping the last snapshot per group
odds_clean = odds.drop_duplicates(subset=dedupe_cols, keep='last')

print("Original rows:", len(odds))
print("Cleaned rows:", len(odds_clean))
print("Duplicates removed:", len(odds) - len(odds_clean))

# Save cleaned file
odds_clean.to_csv("odds_deduped.csv", index=False)


Original rows: 149790
Cleaned rows: 131761
Duplicates removed: 18029


In [99]:
import pandas as pd

# ============================================================
# 1. CONFIG – update these paths if needed
# ============================================================
GAMES_PATH = "nhl_games_with_nst_season_features.csv"
ODDS_PATH = "odds.csv"

DEDUPED_ODDS_PATH = "odds_deduped.csv"
MERGED_OUTPUT_PATH = "games_with_all_books_merged.csv"

# ============================================================
# 2. LOAD DATA
# ============================================================
print("Loading games and odds...")
games = pd.read_csv(GAMES_PATH)
odds = pd.read_csv(ODDS_PATH)

print("Games shape:", games.shape)
print("Odds shape:", odds.shape)

# ============================================================
# 3. CLEAN / DEDUPE ODDS (OPTION 1: KEEP LAST AVAILABLE PRICE)
# ============================================================

# Ensure snapshot_ts is proper datetime so sorting works correctly
if "snapshot_ts" in odds.columns:
    odds["snapshot_ts"] = pd.to_datetime(
        odds["snapshot_ts"],
        utc=True,
        errors="coerce"
    )
else:
    raise ValueError("snapshot_ts column not found in odds.csv")

# Sort so that 'keep=\"last\"' keeps the last (most recent) snapshot
odds = odds.sort_values("snapshot_ts")

# Columns that should uniquely identify a line, adjust if your schema differs
possible_dedupe_cols = [
    "game_date",
    "home_team",
    "away_team",
    "market_type",
    "book",
    "team",        # sometimes 'team' or 'outcome_name' or 'side'
]

dedupe_cols = [c for c in possible_dedupe_cols if c in odds.columns]
print("Using these columns to dedupe odds:", dedupe_cols)

if not dedupe_cols:
    raise ValueError("No dedupe columns found in odds.csv; check column names.")

# Drop duplicates, keeping the LAST row per group (Option 1)
odds_deduped = odds.drop_duplicates(subset=dedupe_cols, keep="last")

print("Original odds rows:", len(odds))
print("Deduped odds rows:", len(odds_deduped))
print("Duplicates removed:", len(odds) - len(odds_deduped))

# Save deduped odds for future use
odds_deduped.to_csv(DEDUPED_ODDS_PATH, index=False)
print(f"Saved deduped odds to: {DEDUPED_ODDS_PATH}")

# ============================================================
# 4. MERGE DEDUPED ODDS ONTO GAMES
# ============================================================

# Make sure game_date is datetime in both for safety
games["game_date"] = pd.to_datetime(games["game_date"])
odds_deduped["game_date"] = pd.to_datetime(odds_deduped["game_date"])

# Merge keys – minimal and robust
merge_keys = [c for c in ["game_date", "home_team", "away_team"] if c in games.columns and c in odds_deduped.columns]
print("Merging on keys:", merge_keys)

if len(merge_keys) < 3:
    raise ValueError("Missing one or more merge keys in either games or odds_deduped.")

merged = games.merge(
    odds_deduped,
    on=merge_keys,
    how="left",   # left join so you keep all games even if some odds are missing
    validate="many_to_many"  # allow multiple markets per game
)

print("Merged shape:", merged.shape)

# Quick sanity check: how many games have at least one odds row?
games_with_any_odds = merged.groupby("game_id")["book"].apply(lambda x: x.notna().any()).mean()
print(f"Proportion of games with at least one book's odds: {games_with_any_odds:.3f}")

# ============================================================
# 5. SAVE MERGED DATASET
# ============================================================

merged.to_csv(MERGED_OUTPUT_PATH, index=False)
print(f"Saved merged game+odds data to: {MERGED_OUTPUT_PATH}")


Loading games and odds...
Games shape: (19118, 95)
Odds shape: (149790, 11)
Using these columns to dedupe odds: ['game_date', 'home_team', 'away_team', 'market_type', 'book', 'team']
Original odds rows: 149790
Deduped odds rows: 131761
Duplicates removed: 18029
Saved deduped odds to: odds_deduped.csv
Merging on keys: ['game_date', 'home_team', 'away_team']
Merged shape: (86062, 103)


/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/1391916041.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  odds_deduped["game_date"] = pd.to_datetime(odds_deduped["game_date"])


Proportion of games with at least one book's odds: 0.226
Saved merged game+odds data to: games_with_all_books_merged.csv


In [105]:
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================
INPUT_PATH = "games_with_all_books_merged.csv"
OUTPUT_PATH = "nhl_modeling_dataset_allbooks_rolling.csv"

# ============================================================
# 1. LOAD DATA
# ============================================================
df = pd.read_csv(INPUT_PATH, low_memory=False)
print("Loaded raw data:", df.shape)

df["game_date"] = pd.to_datetime(df["game_date"])

# Ensure game_id exists
if "game_id" not in df.columns:
    df = df.sort_values(["game_date", "home_team", "away_team"])
    df["game_id"] = np.arange(1, len(df) + 1)

# ============================================================
# 2. BUILD BASE GAME TABLE (ONE ROW PER GAME)
#    Remove odds-specific / snapshot-specific columns
# ============================================================
remove_cols = [
    "snapshot_ts",
    "commence_time",
    "event_id",
    "book",
    "market_type",
    "team",
    "odds",
    "point",
]

base_cols = [c for c in df.columns if c not in remove_cols]
games_base = df[base_cols].drop_duplicates(subset=["game_id"]).copy()
print("Base games (one row per game):", games_base.shape)

# ============================================================
# 3. BUILD WIDE ODDS TABLE (ALL BOOKS, ML + SPREADS + TOTALS)
# ============================================================
odds_cols_needed = [
    "game_id",
    "game_date",
    "home_team",
    "away_team",
    "book",
    "market_type",
    "team",
    "odds",
    "point",
]
odds_cols_available = [c for c in odds_cols_needed if c in df.columns]
odds = df[odds_cols_available].copy()

# Normalize book names
odds["book_clean"] = odds["book"].astype(str).str.lower().str.replace(" ", "_")

def map_side(row):
    if row["team"] == row["home_team"]:
        return "home"
    elif row["team"] == row["away_team"]:
        return "away"
    else:
        return np.nan

# -----------------------------
# 3.1 MONEYLINES (H2H)
# -----------------------------
ml = odds[odds["market_type"] == "h2h"].copy()
ml["side"] = ml.apply(map_side, axis=1)
ml = ml[ml["side"].isin(["home", "away"])].copy()
ml["col_name"] = ml["book_clean"] + "_ml_" + ml["side"]

ml_wide = (
    ml.pivot_table(
        index="game_id",
        columns="col_name",
        values="odds",
        aggfunc="last",
    )
    .reset_index()
)
ml_wide.columns = [str(c) for c in ml_wide.columns]
print("Wide moneyline odds:", ml_wide.shape)

# -----------------------------
# 3.2 SPREADS (PUCKLINE)
# -----------------------------
spreads = odds[odds["market_type"] == "spreads"].copy()
if not spreads.empty:
    spreads["side"] = spreads.apply(map_side, axis=1)
    spreads = spreads[spreads["side"].isin(["home", "away"])].copy()
    spreads["line_col"] = spreads["book_clean"] + "_spread_" + spreads["side"] + "_line"
    spreads["odds_col"] = spreads["book_clean"] + "_spread_" + spreads["side"] + "_odds"

    spread_line_wide = (
        spreads.pivot_table(
            index="game_id",
            columns="line_col",
            values="point",
            aggfunc="last",
        )
        .reset_index()
    )
    spread_line_wide.columns = [str(c) for c in spread_line_wide.columns]

    spread_odds_wide = (
        spreads.pivot_table(
            index="game_id",
            columns="odds_col",
            values="odds",
            aggfunc="last",
        )
        .reset_index()
    )
    spread_odds_wide.columns = [str(c) for c in spread_odds_wide.columns]

    print("Wide spread lines:", spread_line_wide.shape)
    print("Wide spread odds:", spread_odds_wide.shape)
else:
    spread_line_wide = pd.DataFrame({"game_id": games_base["game_id"].unique()})
    spread_odds_wide = spread_line_wide.copy()
    print("No spread data found.")

# -----------------------------
# 3.3 TOTALS (OVER / UNDER)
# -----------------------------
totals = odds[odds["market_type"] == "totals"].copy()
if not totals.empty:
    totals["direction"] = (
        totals["team"]
        .astype(str)
        .str.lower()
        .str.extract("(over|under)", expand=False)
    )
    totals = totals[totals["direction"].isin(["over", "under"])].copy()
    totals["odds_col"] = (
        totals["book_clean"] + "_total_" + totals["direction"] + "_odds"
    )

    totals_odds_wide = (
        totals.pivot_table(
            index="game_id",
            columns="odds_col",
            values="odds",
            aggfunc="last",
        )
        .reset_index()
    )
    totals_odds_wide.columns = [str(c) for c in totals_odds_wide.columns]

    totals_line = (
        totals.groupby(["game_id", "book_clean"])["point"]
        .median()
        .reset_index()
    )
    totals_line["line_col"] = totals_line["book_clean"] + "_total_line"

    totals_line_wide = (
        totals_line.pivot_table(
            index="game_id",
            columns="line_col",
            values="point",
            aggfunc="last",
        )
        .reset_index()
    )
    totals_line_wide.columns = [str(c) for c in totals_line_wide.columns]

    print("Wide totals line:", totals_line_wide.shape)
    print("Wide totals odds:", totals_odds_wide.shape)
else:
    totals_line_wide = pd.DataFrame({"game_id": games_base["game_id"].unique()})
    totals_odds_wide = totals_line_wide.copy()
    print("No totals data found.")

# -----------------------------
# 3.4 COMBINE ALL ODDS
# -----------------------------
odds_wide = ml_wide
for tbl in [spread_line_wide, spread_odds_wide, totals_line_wide, totals_odds_wide]:
    odds_wide = odds_wide.merge(tbl, on="game_id", how="left")

print("Combined wide odds table:", odds_wide.shape)

# -----------------------------
# 3.5 IMPLIED PROBABILITIES FOR MONEYLINES
# -----------------------------
def american_to_prob(o):
    try:
        o = float(o)
    except (TypeError, ValueError):
        return np.nan
    if o < 0:
        return -o / (-o + 100.0)
    else:
        return 100.0 / (o + 100.0)

for col in odds_wide.columns:
    if col.endswith("_ml_home") or col.endswith("_ml_away"):
        probs_col = col + "_prob"
        odds_wide[probs_col] = odds_wide[col].apply(american_to_prob)

# Merge odds into base games
games = games_base.merge(odds_wide, on="game_id", how="left")
print("Games with all odds merged:", games.shape)

# ============================================================
# 4. BUILD LONG TEAM-GAME TABLE FOR ROLLING FEATURES
# ============================================================
home_cols = [c for c in games.columns if c.startswith("home_")]
away_cols = [c for c in games.columns if c.startswith("away_")]

home_stats = [c.replace("home_", "") for c in home_cols]
away_stats = [c.replace("away_", "") for c in away_cols]
common_stats = sorted(set(home_stats) & set(away_stats))

# IMPORTANT: remove non-metric identifiers like "team" and "team_code"
common_stats = [s for s in common_stats if s not in ["team", "team_code"]]

print("Common home/away stats used for rolling:", len(common_stats))

meta_cols = ["game_id", "game_date"]
if "season" in games.columns:
    meta_cols.append("season")

home_long_cols = meta_cols + ["home_team"] + [f"home_{s}" for s in common_stats]
away_long_cols = meta_cols + ["away_team"] + [f"away_{s}" for s in common_stats]

home_long = games[home_long_cols].copy()
home_long = home_long.rename(columns={"home_team": "team"})
home_long["is_home"] = 1
home_long = home_long.rename(columns={f"home_{s}": s for s in common_stats})

away_long = games[away_long_cols].copy()
away_long = away_long.rename(columns={"away_team": "team"})
away_long["is_home"] = 0
away_long = away_long.rename(columns={f"away_{s}": s for s in common_stats})

long_df = pd.concat([home_long, away_long], ignore_index=True)
print("Long team-game frame:", long_df.shape)

# Sort by team/season/date
group_keys = ["team"]
if "season" in long_df.columns:
    group_keys.append("season")

long_df = long_df.sort_values(group_keys + ["game_date"])

# ============================================================
# 5. DEFINE STATS + WINDOWS FOR ROLLING / EWM
# ============================================================
preferred_stats = [
    "xGF", "xGA", "xGFPct",
    "CF", "CA", "CFPct",
    "HDCF", "HDCA", "HDCFPct",
    "GF", "GA",
    "SFPct",
    "SVPct",
    "PDO",
]
stats_available = [s for s in preferred_stats if s in long_df.columns]
print("Stats available for rolling:", stats_available)

windows = [3, 5, 10, 20]    # short, medium, form, long
ewma_alphas = [0.2, 0.05]   # fast + medium EWMA

# ============================================================
# 6. ROLLING MEANS & EWMAS
# ============================================================
for stat in stats_available:
    lag_col = f"{stat}_lag1"
    long_df[lag_col] = long_df.groupby(group_keys)[stat].shift(1)

    for w in windows:
        roll_col = f"{stat}_roll{w}"
        long_df[roll_col] = (
            long_df
            .groupby(group_keys)[lag_col]
            .transform(lambda s: s.rolling(window=w, min_periods=1).mean())
        )

    for alpha in ewma_alphas:
        ewm_col = f"{stat}_ewm{str(alpha).replace('.', '')}"
        long_df[ewm_col] = (
            long_df
            .groupby(group_keys)[lag_col]
            .transform(lambda s: s.ewm(alpha=alpha, adjust=False).mean())
        )

# ============================================================
# 7. GOAL DIFF & WIN% ROLLING
# ============================================================
if {"GF", "GA"}.issubset(long_df.columns):
    long_df["goals_for"] = long_df["GF"]
    long_df["goals_against"] = long_df["GA"]
    long_df["goal_diff"] = long_df["goals_for"] - long_df["goals_against"]
    long_df["win_flag"] = (long_df["goal_diff"] > 0).astype(int)

    for w in [5, 10, 20]:
        gd_col = f"goal_diff_roll{w}"
        wp_col = f"win_pct_roll{w}"

        long_df[gd_col] = (
            long_df
            .groupby(group_keys)["goal_diff"]
            .transform(lambda s: s.shift(1).rolling(window=w, min_periods=1).mean())
        )

        long_df[wp_col] = (
            long_df
            .groupby(group_keys)["win_flag"]
            .transform(lambda s: s.shift(1).rolling(window=w, min_periods=1).mean())
        )

# ============================================================
# 8. TREND FEATURES (SHORT VS LONG FORM)
# ============================================================
trend_stats = [s for s in ["xGFPct", "CFPct", "HDCFPct", "PDO"] if s in stats_available]

for stat in trend_stats:
    r3 = f"{stat}_roll3"
    r5 = f"{stat}_roll5"
    r10 = f"{stat}_roll10"
    r20 = f"{stat}_roll20"

    if r5 in long_df.columns and r20 in long_df.columns:
        long_df[f"{stat}_trend_5v20"] = long_df[r5] - long_df[r20]

    if r3 in long_df.columns and r10 in long_df.columns:
        long_df[f"{stat}_trend_3v10"] = long_df[r3] - long_df[r10]

# ============================================================
# 9. SPLIT HOME / AWAY FEATURES & MERGE BACK TO GAME-LEVEL
# ============================================================
feature_cols = [
    c for c in long_df.columns
    if any(
        tag in c
        for tag in [
            "_roll3", "_roll5", "_roll10", "_roll20",
            "_ewm", "trend_5v20", "trend_3v10",
            "goal_diff_roll5", "goal_diff_roll10", "goal_diff_roll20",
            "win_pct_roll5", "win_pct_roll10", "win_pct_roll20",
        ]
    )
]
feature_cols = ["game_id", "is_home"] + feature_cols

features_long = long_df[feature_cols].copy()
home_features_long = features_long[features_long["is_home"] == 1].drop(columns=["is_home"])
away_features_long = features_long[features_long["is_home"] == 0].drop(columns=["is_home"])

home_features_long = home_features_long.rename(
    columns={c: f"home_{c}" for c in home_features_long.columns if c != "game_id"}
)
away_features_long = away_features_long.rename(
    columns={c: f"away_{c}" for c in away_features_long.columns if c != "game_id"}
)

games_roll = games.merge(home_features_long, on="game_id", how="left")
games_roll = games_roll.merge(away_features_long, on="game_id", how="left")
print("Games with rolling features:", games_roll.shape)

# ============================================================
# 10. OPPONENT-ADJUSTED FEATURES (HOME − AWAY)
# ============================================================
core_diff_bases = [
    "xGFPct_roll10",
    "CFPct_roll10",
    "HDCFPct_roll10",
    "goal_diff_roll10",
    "win_pct_roll10",
]

for base in core_diff_bases:
    home_col = f"home_{base}"
    away_col = f"away_{base}"
    if home_col in games_roll.columns and away_col in games_roll.columns:
        games_roll[f"diff_{base}"] = games_roll[home_col] - games_roll[away_col]

print("Opponent-adjusted features added:",
      [c for c in games_roll.columns if c.startswith("diff_")])

# ============================================================
# 11. SAVE FINAL MODELING DATASET
# ============================================================
games_roll.to_csv(OUTPUT_PATH, index=False)
print("Saved final modeling dataset to:", OUTPUT_PATH)
print("Final shape:", games_roll.shape)


Loaded raw data: (86062, 103)
Base games (one row per game): (19118, 95)
Wide moneyline odds: (4308, 7)
Wide spread lines: (4294, 7)
Wide spread odds: (4294, 7)
Wide totals line: (4305, 4)
Wide totals odds: (4305, 7)
Combined wide odds table: (4308, 28)
Games with all odds merged: (19118, 128)
Common home/away stats used for rolling: 43
Long team-game frame: (38236, 48)
Stats available for rolling: ['xGF', 'xGA', 'xGFPct', 'CF', 'CA', 'CFPct', 'HDCF', 'HDCA', 'HDCFPct', 'GF', 'GA', 'SFPct', 'SVPct', 'PDO']


/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/3816179174.py:281: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  long_df[roll_col] = (
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/3816179174.py:281: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  long_df[roll_col] = (
/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_97355/3816179174.py:281: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance. 

Games with rolling features: (19118, 324)
Opponent-adjusted features added: ['diff_xGFPct_roll10', 'diff_CFPct_roll10', 'diff_HDCFPct_roll10', 'diff_goal_diff_roll10', 'diff_win_pct_roll10']
Saved final modeling dataset to: nhl_modeling_dataset_allbooks_rolling.csv
Final shape: (19118, 329)


In [107]:
df = pd.read_csv("nhl_modeling_dataset_allbooks_rolling.csv", low_memory=False)

print("Shape:", df.shape)
print("Unique game_ids:", df["game_id"].nunique())

dupes = df[df.duplicated("game_id", keep=False)]
print("Duplicate game_id rows:", dupes.shape[0])


Shape: (19118, 329)
Unique game_ids: 19118
Duplicate game_id rows: 0


In [109]:
core_cols = [
    "home_goals", "away_goals", "home_win",
    "home_xGFPct_roll10", "away_xGFPct_roll10",
    "diff_xGFPct_roll10", "pinnacle_ml_home_prob"
]

missing_report = df[core_cols].isna().sum()
print("Missing core fields:\n", missing_report)


Missing core fields:
 home_goals                   0
away_goals                   0
home_win                     0
home_xGFPct_roll10        2865
away_xGFPct_roll10        2842
diff_xGFPct_roll10        5237
pinnacle_ml_home_prob    14952
dtype: int64


In [111]:
books = [c for c in df.columns if "_ml_home" in c]

for b in books:
    print(b, df[b].notna().mean())


draftkings_ml_home 0.21879903755622973
fanduel_ml_home 0.22204205460822263
pinnacle_ml_home 0.21790982320326394
draftkings_ml_home_prob 0.21879903755622973
fanduel_ml_home_prob 0.22204205460822263
pinnacle_ml_home_prob 0.21790982320326394


In [113]:
bad_rows = df[
    (df["pinnacle_ml_home"] > 0) & (df["pinnacle_ml_away"] > 0)
]
print("Invalid Pinnacle odds:", bad_rows.shape[0])


Invalid Pinnacle odds: 25


In [115]:
probs = [c for c in df.columns if "_prob" in c]

for p in probs:
    print(p, df[p].min(), df[p].max())


draftkings_ml_away_prob 0.0384615384615384 0.999000999000999
draftkings_ml_home_prob 0.024390243902439 0.999000999000999
fanduel_ml_away_prob 0.0384615384615384 0.9950248756218906
fanduel_ml_home_prob 0.0476190476190476 0.9950248756218906
pinnacle_ml_away_prob 0.0745712155108128 0.9705275567344532
pinnacle_ml_home_prob 0.054054054054054 0.9570815450643776


In [117]:
roll_cols = [c for c in df.columns if "roll10" in c]

df[roll_cols].describe().T.head(20)


,count,mean,std,min,25%,50%,75%,max
home_xGF_roll10,16253.0,216.203386,40.720471,102.32,202.160,221.190,241.32,310.93
home_xGA_roll10,16253.0,216.561005,39.701852,99.50,201.920,223.120,240.83,322.95
home_xGFPct_roll10,16253.0,49.940022,3.217500,39.72,47.910,50.110,52.11,59.67
home_CF_roll10,16253.0,4405.384175,725.971169,2284.00,4218.000,4595.000,4860.00,5821.00
home_CA_roll10,16253.0,4414.920692,713.390388,2328.00,4275.000,4617.000,4858.00,5629.00
home_CFPct_roll10,16253.0,49.932626,2.772816,37.97,48.190,49.890,51.73,59.87
home_HDCF_roll10,16253.0,855.786624,168.751327,406.00,782.000,870.000,965.00,1301.00
home_HDCA_roll10,16253.0,860.546484,162.580453,389.00,791.000,887.000,958.00,1357.00
home_HDCFPct_roll10,16253.0,49.831884,3.566216,39.28,47.650,49.870,52.25,59.34
home_GF_roll10,16253.0,222.345721,41.814167,109.00,206.000,226.000,250.00,337.00


In [119]:
diff_cols = [c for c in df.columns if c.startswith("diff_")]

df[diff_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
diff_xGFPct_roll10,13881.0,0.003229,4.658347,-17.19,-3.08,0.01,3.09,17.19
diff_CFPct_roll10,13881.0,0.000656,4.019015,-16.81,-2.55,-0.01,2.53,16.81
diff_HDCFPct_roll10,13881.0,0.002575,5.167649,-17.49,-3.51,0.00,3.50,17.49
diff_goal_diff_roll10,13881.0,0.065845,59.055017,-256.00,-38.00,0.00,38.00,256.00
diff_win_pct_roll10,18844.0,0.001592,0.713772,-1.00,-1.00,0.00,1.00,1.00


In [121]:
df[["home_xGFPct_roll10", "home_win"]].corr()
df[["away_xGFPct_roll10", "home_win"]].corr()


,away_xGFPct_roll10,home_win
away_xGFPct_roll10,1.000000,-0.091335
home_win,-0.091335,1.000000


In [147]:
import pandas as pd
import numpy as np

# =========================================
# 1. Load the full rolling+odds dataset
# =========================================
full_path = "nhl_modeling_dataset_allbooks_rolling.csv"
df = pd.read_csv(full_path, low_memory=False)

print("Full dataset shape:", df.shape)
print("Date range:", df["game_date"].min(), "->", df["game_date"].max())

# Ensure datetime
df["game_date"] = pd.to_datetime(df["game_date"])

# =========================================
# 2. Filter to rows with Pinnacle ML odds
# =========================================
required_odds_cols = ["pinnacle_ml_home", "pinnacle_ml_away"]

for col in required_odds_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required odds column: {col}")

mask_pinnacle_present = df["pinnacle_ml_home"].notna() & df["pinnacle_ml_away"].notna()
df_pinn = df[mask_pinnacle_present].copy()

print("Rows with Pinnacle ML both sides:", df_pinn.shape[0])

# =========================================
# 3. Drop invalid Pinnacle odds
#    (both sides positive is not a valid 2-way market)
# =========================================
invalid_mask = (df_pinn["pinnacle_ml_home"] > 0) & (df_pinn["pinnacle_ml_away"] > 0)
print("Invalid Pinnacle odds rows (both positive):", invalid_mask.sum())

df_pinn = df_pinn[~invalid_mask].copy()

# (Optionally you can also guard against both negative, but that's rare)
both_negative = (df_pinn["pinnacle_ml_home"] < 0) & (df_pinn["pinnacle_ml_away"] < 0)
print("Both negative Pinnacle odds rows:", both_negative.sum())

# =========================================
# 4. Filter to rows with complete key rolling features
# =========================================
key_roll_cols = [
    "home_xGFPct_roll10",
    "away_xGFPct_roll10",
    "diff_xGFPct_roll10",
    "home_CFPct_roll10",
    "away_CFPct_roll10",
    "diff_CFPct_roll10",
    "home_HDCFPct_roll10",
    "away_HDCFPct_roll10",
    "diff_HDCFPct_roll10",
    "home_goal_diff_roll10",
    "away_goal_diff_roll10",
    "diff_goal_diff_roll10",
    "home_win_pct_roll10",
    "away_win_pct_roll10",
    "diff_win_pct_roll10"
]

existing_key_roll_cols = [c for c in key_roll_cols if c in df_pinn.columns]
print("Key rolling columns used for completeness filter:", existing_key_roll_cols)

mask_complete_roll = df_pinn[existing_key_roll_cols].notna().all(axis=1)
print("Rows with complete key rolling features:", mask_complete_roll.sum())

df_model_ready = df_pinn[mask_complete_roll].copy()

print("Model-ready dataset shape:", df_model_ready.shape)
print("Model-ready date range:",
      df_model_ready["game_date"].min(), "->", df_model_ready["game_date"].max())

# =========================================
# 5. (Optional) sanity: check odds coverage of other books
# =========================================
book_ml_cols = [c for c in df_model_ready.columns if c.endswith("_ml_home")]
print("\nCoverage in model-ready set:")
for col in book_ml_cols:
    print(col, "coverage:", df_model_ready[col].notna().mean())

# =========================================
# 6. Save model-ready dataset
# =========================================
out_path = "nhl_modeling_dataset_allbooks_rolling_modelready.csv"
df_model_ready.to_csv(out_path, index=False)
print("\nSaved model-ready dataset to:", out_path)


Full dataset shape: (19118, 329)
Date range: 2009-10-01 -> 2024-06-24
Rows with Pinnacle ML both sides: 4166
Invalid Pinnacle odds rows (both positive): 25
Both negative Pinnacle odds rows: 297
Key rolling columns used for completeness filter: ['home_xGFPct_roll10', 'away_xGFPct_roll10', 'diff_xGFPct_roll10', 'home_CFPct_roll10', 'away_CFPct_roll10', 'diff_CFPct_roll10', 'home_HDCFPct_roll10', 'away_HDCFPct_roll10', 'diff_HDCFPct_roll10', 'home_goal_diff_roll10', 'away_goal_diff_roll10', 'diff_goal_diff_roll10', 'home_win_pct_roll10', 'away_win_pct_roll10', 'diff_win_pct_roll10']
Rows with complete key rolling features: 2951
Model-ready dataset shape: (2951, 329)
Model-ready date range: 2020-08-03 00:00:00 -> 2024-06-24 00:00:00

Coverage in model-ready set:
draftkings_ml_home coverage: 0.973568281938326
fanduel_ml_home coverage: 0.9932226363944425
pinnacle_ml_home coverage: 1.0

Saved model-ready dataset to: nhl_modeling_dataset_allbooks_rolling_modelready.csv


In [151]:
df = pd.read_csv('nhl_modeling_dataset_allbooks_rolling_modelready.csv')

In [155]:
df.game_date.min(), df.game_date.max()

('2020-08-03', '2024-06-24')

In [139]:
odds = pd.read_csv('odds_deduped.csv')

In [141]:
odds.market_type.unique()

array(['h2h', 'spreads', 'totals'], dtype=object)

In [143]:
odds.head()

,snapshot_ts,game_date,commence_time,event_id,home_team,away_team,book,market_type,team,odds,point
0,2020-08-01 23:55:00+00:00,2020-08-01,2020-08-02 02:30:00+00:00,b2020f52aad491b409755933742b3603,Calgary Flames,Winnipeg Jets,draftkings,h2h,Calgary Flames,-127,NaN
1,2020-08-01 23:55:00+00:00,2020-08-02,2020-08-02 19:00:00+00:00,02abf721b0e5ce346ac52f438fc407de,Boston Bruins,Philadelphia Flyers,pinnacle,h2h,Boston Bruins,115,NaN
2,2020-08-01 23:55:00+00:00,2020-08-02,2020-08-02 19:00:00+00:00,02abf721b0e5ce346ac52f438fc407de,Boston Bruins,Philadelphia Flyers,pinnacle,h2h,Philadelphia Flyers,193,NaN
3,2020-08-01 23:55:00+00:00,2020-08-02,2020-08-02 19:00:00+00:00,02abf721b0e5ce346ac52f438fc407de,Boston Bruins,Philadelphia Flyers,pinnacle,h2h,Draw,307,NaN
4,2020-08-01 23:55:00+00:00,2020-08-02,2020-08-02 19:00:00+00:00,02abf721b0e5ce346ac52f438fc407de,Boston Bruins,Philadelphia Flyers,pinnacle,spreads,Boston Bruins,115,-0.5


In [145]:
odds[(odds.game_date == '2021-01-13') & (odds.home_team == 'Philadelphia Flyers')][['game_date','home_team','market_type','book','odds']]

,game_date,home_team,market_type,book,odds
1196,2021-01-13,Philadelphia Flyers,h2h,fanduel,-105
1198,2021-01-13,Philadelphia Flyers,h2h,fanduel,-115
1199,2021-01-13,Philadelphia Flyers,h2h,draftkings,-120
1204,2021-01-13,Philadelphia Flyers,h2h,draftkings,-105


In [125]:
df[df.season>2020][['season','game_date','home_team','pinnacle_ml_home','fanduel_ml_home']]

,season,game_date,home_team,pinnacle_ml_home,fanduel_ml_home
13965,2021,2021-01-13,Colorado Avalanche,NaN,NaN
13966,2021,2021-01-13,Edmonton Oilers,-141.0,-135.0
13967,2021,2021-01-13,Philadelphia Flyers,NaN,-105.0
13968,2021,2021-01-13,Tampa Bay Lightning,-250.0,-278.0
13969,2021,2021-01-13,Toronto Maple Leafs,NaN,NaN
...,...,...,...,...,...
19113,2024,2024-06-13,Edmonton Oilers,-139.0,-138.0
19114,2024,2024-06-15,Edmonton Oilers,-117.0,-120.0
19115,2024,2024-06-18,Florida Panthers,-149.0,-154.0
19116,2024,2024-06-21,Edmonton Oilers,-115.0,-120.0
